<a href="https://colab.research.google.com/github/FengruiJing/Multi-network-spatial-error-framework/blob/main/Multi_network_spatial_error_framework_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#U.S. data and codes
shp_path = "US_HIV_Merged_total.shp"

Added 2018 STDS
excel_path = "2018Rates1.xlsx"

Download link:
https://drive.google.com/file/d/1-1HNcQqbDhUkA-33tdOnYuDYUJIYCkLY/view?usp=sharing

https://docs.google.com/spreadsheets/d/1cpXbaAEpu7A3Xy0_pjD0FsaJYjxGUizG/edit?usp=sharing&ouid=118218235341025234377&rtpof=true&sd=true

Twitter-based physical connectivity data are available from https://github.com/GIBDUSC/Place-Connectivity-Index.

Facebook-based social connectivity data are available from https://dataforgood.facebook.com/dfg/tools/social-connectedness-index.

The results of Table 2 and Figure 3 can be generated by the data and codes.

#Link2018STDsData

In [ ]:
"""
Merge 2018 STI rate variables from an Excel file into one or more county-level
geospatial files (e.g., shapefiles), drop rows with missing required fields,
and save cleaned outputs.

Notes
-----
- FIPS should be treated as a 5-digit string. This script normalizes FIPS to
  digits-only and zero-pads to length 5 (e.g., "01001").
- If you output ESRI Shapefile (*.shp), be aware of field name length limits
  (often 10 characters). Consider GeoPackage (*.gpkg) for safer column names.
"""

from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
import geopandas as gpd


# Columns expected in the shapefiles (kept if present)
BASE_COLS: List[str] = [
    "GonorrheaR", "ChlamydiaR", "PercenHIVp",
    "Population", "UrbanRural", "Female", "Old", "Black", "Noinsuranc",
    "Poverty", "crime16to1", "Dissimilar",
]

# Columns expected in the Excel file
NEW_RATE_COLS: List[str] = ["RateChlamydia2018", "RateGonorrhea2018", "RateHIV2018"]

# Rows are dropped if any of these fields are missing after merge
REQUIRED_COLS: List[str] = ["GonorrheaR", "ChlamydiaR", "PercenHIVp"] + NEW_RATE_COLS


def normalize_fips(x) -> Optional[str]:
    """
    Normalize a FIPS code to a 5-digit string:
    - Keep digits only
    - Zero-pad to length 5
    Returns None if input is missing or cannot be parsed.
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    digits = re.sub(r"\D+", "", s)
    if digits == "":
        return None
    # Some sources may provide longer codes; keep the last 5 digits if needed.
    if len(digits) > 5:
        digits = digits[-5:]
    return digits.zfill(5)


def load_rates(excel_path: str) -> pd.DataFrame:
    """Load and standardize the Excel rates table."""
    df = pd.read_excel(excel_path)

    if "FIPS" not in df.columns:
        raise KeyError("Excel file must contain a 'FIPS' column.")

    # Normalize FIPS and keep only required columns
    df["FIPS"] = df["FIPS"].apply(normalize_fips)
    keep_cols = ["FIPS"] + NEW_RATE_COLS

    missing_cols = [c for c in keep_cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Excel file is missing required columns: {missing_cols}")

    df = df[keep_cols].copy()
    df = df.dropna(subset=["FIPS"])
    df = df.drop_duplicates(subset=["FIPS"])

    return df


def merge_and_clean(
    shp_in: str,
    shp_out: str,
    rates_df: pd.DataFrame,
    verbose: bool = True,
) -> gpd.GeoDataFrame:
    """
    Read a geospatial file, merge 2018 rate columns by FIPS, drop rows with
    missing REQUIRED_COLS, keep only selected columns, and write output.
    """
    if verbose:
        print("\n========================================")
        print(f"Processing: {shp_in}")
        print("========================================")

    gdf = gpd.read_file(shp_in)

    if "FIPS" not in gdf.columns:
        raise KeyError(f"Input file '{shp_in}' must contain a 'FIPS' column.")

    gdf = gdf.copy()
    gdf["FIPS"] = gdf["FIPS"].apply(normalize_fips)
    gdf = gdf.dropna(subset=["FIPS"])
    gdf = gdf.drop_duplicates(subset=["FIPS"])

    # Left join: keep all geometries from the input file
    merged = gdf.merge(rates_df, on="FIPS", how="left")
    merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=gdf.crs)

    # Ensure REQUIRED_COLS exist after merge
    for col in REQUIRED_COLS:
        if col not in merged.columns:
            raise KeyError(f"Required column '{col}' not found after merge for '{shp_in}'.")

    if verbose:
        miss_counts = merged[REQUIRED_COLS].isna().sum()
        miss_pct = (merged[REQUIRED_COLS].isna().mean() * 100).round(2)

        print("[After merge] Missing counts (REQUIRED_COLS):")
        print(miss_counts.to_string())
        print("\n[After merge] Missing percentages (REQUIRED_COLS):")
        print(miss_pct.astype(str).add("%").to_string())

    cleaned = merged.dropna(subset=REQUIRED_COLS)

    if verbose:
        print(f"\nRows before dropna: {len(merged)}, after dropna: {len(cleaned)}")

    # Keep only selected columns that exist
    keep_cols = ["FIPS"] + BASE_COLS + NEW_RATE_COLS + ["geometry"]
    keep_cols = [c for c in keep_cols if c in cleaned.columns]
    final = cleaned[keep_cols].copy()

    if verbose:
        print("\n[Final columns]")
        print(list(final.columns))

    # Write output (driver inferred from extension)
    final.to_file(shp_out)

    if verbose:
        print(f"\nSaved: {shp_out}")

    return final


def main():
    parser = argparse.ArgumentParser(
        description="Merge 2018 rates from Excel into geospatial files by FIPS and export cleaned outputs."
    )
    parser.add_argument("--excel", default="2018Rates1.xlsx", help="Path to the Excel file containing 2018 rates.")
    parser.add_argument(
        "--pairs",
        nargs="+",
        default=[
            "US_HIV_Merged_total.shp:US_HIV_Merged_total_final.shp",
            "DeepSouth_HIV_Merged.shp:DeepSouth_HIV_Merged_final.shp",
            "CBSA_HIV_Merged.shp:CBSA_HIV_Merged_final.shp",
        ],
        help="Input:output pairs, e.g., in1.shp:out1.shp in2.shp:out2.shp",
    )
    parser.add_argument("--quiet", action="store_true", help="Reduce console output.")
    args = parser.parse_args()

    rates = load_rates(args.excel)

    verbose = not args.quiet
    for pair in args.pairs:
        if ":" not in pair:
            raise ValueError(f"Invalid pair '{pair}'. Use 'input:output'.")
        shp_in, shp_out = pair.split(":", 1)
        merge_and_clean(shp_in=shp_in, shp_out=shp_out, rates_df=rates, verbose=verbose)

    if verbose:
        print("\nDone. All files processed and saved.")


if __name__ == "__main__":
    main()

#ModelRunning

In [ ]:
!pip install pysal

In [ ]:
!pip install esda

In [ ]:
#mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os
os.chdir("/content/gdrive/My Drive/Colab Notebooks")

##Disease types

For different diseases, simply modify the dependent variable assignment in the first two codes for the main workflow.

If you are modeling HIV prevalence, use:

STD_Y_COL = 'PercenHIVp'

If you are modeling Gonorrhea rates, use:

STD_Y_COL = 'GonorrheaR'

If you are modeling Chlamydia rates, use:

STD_Y_COL = 'ChlamydiaR'


##Main workflow

In [ ]:
# ============================================
# 0. Imports & Global config
# ============================================

import os
import math
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from joblib import Parallel, delayed

from libpysal.weights import Queen, W
from libpysal import weights
from esda.moran import Moran
from spreg.diagnostics import log_likelihood, aic

from numpy.linalg import inv, slogdet
from scipy.optimize import minimize

from esda.moran import Moran_Local  # optional (not required)
from shapely.geometry import Point

warnings.filterwarnings("ignore")

# ------------------------------
# Global random seed
# ------------------------------
RANDOM_SEED = 7
rng_global = np.random.default_rng(RANDOM_SEED)

# Parallel / production mode
N_JOBS = -1        # -1: all cores; set to 1 if unstable
RUN_PROD = False   # If True, you may increase iterations / B_NULL / B_PERT, etc.

# ------------- Outcome config: switch STD outcome in one line ----------------
# Options: 'PercenHIVp', 'GonorrheaR', 'ChlamydiaR'
STD_Y_COL = 'PercenHIVp'   # <--- change this line only

# ------------------------------
# Output folder
# ------------------------------
OUTDIR = "./US_MAIN_ANALYSIS_OUTPUT"
os.makedirs(OUTDIR, exist_ok=True)

# Mapping to 2018 field names for temporal robustness
TEMPORAL_2018_COL = {
    'PercenHIVp': 'RateHIV2018',
    'GonorrheaR': 'RateGonorrhea2018',
    'ChlamydiaR': 'RateChlamydia2018',
}

# ============================================
# 1. Read US shapefile & basic preprocessing
# ============================================

# 1.1 Read US shapefile (field structure consistent with DeepSouth)
shp_path = "US_HIV_Merged_total_final.shp"

gdf = gpd.read_file(shp_path)
if 'FIPS' not in gdf.columns:
    raise ValueError("FIPS column not found in shapefile.")

# Strip leading zeros from FIPS and drop duplicates
gdf['FIPS'] = gdf['FIPS'].astype(str).str.zfill(5)
gdf = gdf.drop_duplicates(subset=['FIPS']).copy()

# Fix potentially truncated 2018 field names
# (In some systems, shapefile field names can be truncated to 10 chars)
# We handle known mappings here if needed:
rename_map_2018 = {
    'RateHIV201': 'RateHIV2018',
    'RateGonor': 'RateGonorrhea2018',
    'RateChlam': 'RateChlamydia2018',
}
for k, v in rename_map_2018.items():
    if k in gdf.columns and v not in gdf.columns:
        gdf = gdf.rename(columns={k: v})

# Covariates
COVARS = [
    'Poverty',
    'Black',
    'Hispanic',
    'Age1524',
    'Urban',
    'Uninsured',
]

# Drop missing values for the current 2019 outcome + covariates
need_cols = [STD_Y_COL] + COVARS
gdf = gdf.dropna(subset=need_cols).copy()

# State code
if 'STATEFP' in gdf.columns:
    gdf['STATE'] = gdf['STATEFP'].astype(str).str.zfill(2)
elif 'STATE' not in gdf.columns:
    raise ValueError("STATE code not found (expected STATEFP or STATE).")

# 1.2 Simple interannual correlation: 2018 vs 2019 (US)
def _print_temporal_corr(col_2019, col_2018, label):
    if {col_2019, col_2018}.issubset(gdf.columns):
        sub = gdf[[col_2019, col_2018]].dropna()
        if len(sub) > 10:
            corr = sub[col_2019].corr(sub[col_2018])
            print(f"[Temporal Corr] {label}: corr({col_2019},{col_2018}) = {corr:.4f}")
        else:
            print(f"[Temporal Corr] {label}: insufficient overlap after dropna.")
    else:
        print(f"[Temporal Corr] {label}: missing columns.")

y2018_col = TEMPORAL_2018_COL.get(STD_Y_COL, None)
if y2018_col is not None:
    _print_temporal_corr(STD_Y_COL, y2018_col, "US full sample")

# ============================================
# 2. Build W matrices: WP / WS / WQ
# ============================================

# Load precomputed WP / WS from the shapefile (assumed in columns)
# Example: WP and WS are stored as adjacency matrices in external files or as attributes.
# Here we assume WP and WS are read from external CSV/NPY for US main analysis.
# Replace paths if needed:

WP_PATH = "WP_matrix.npy"
WS_PATH = "WS_matrix.npy"

if not os.path.exists(WP_PATH):
    raise ValueError(f"WP matrix not found: {WP_PATH}")
if not os.path.exists(WS_PATH):
    raise ValueError(f"WS matrix not found: {WS_PATH}")

WP_full = np.load(WP_PATH)  # assumed already aligned to gdf ordering OR we will align by IDs
WS_full = np.load(WS_PATH)

# 2.1 Build IDs and alignment index
ids = list(gdf['FIPS'].values)
N = len(ids)

# Sanity check for matrix shapes
if WP_full.shape[0] != WP_full.shape[1] or WS_full.shape[0] != WS_full.shape[1]:
    raise ValueError("WP/WS matrices must be square.")
if WP_full.shape[0] != N or WS_full.shape[0] != N:
    raise ValueError("WP/WS matrices shape does not match gdf sample size. Please align first.")

# 2.2 WP: symmetrize + row-standardize
A = WP_full.astype(float)
A = A + A.T - np.diag(np.diag(A))        # Symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP = A / row_sum                         # Row-standardize

# 2.3 WQ: queen contiguity (US)
# Ensure geometry is valid
gdf = gdf[gdf.geometry.notnull()].copy()
gdf = gdf.set_index('FIPS')

wQ = Queen.from_dataframe(gdf, ids=list(gdf.index))
# Row-standardize libpysal W object
wQ.transform = 'R'

# 2.4 Align common IDs & order (WP / WS / WQ / df_geo consistent)
# If gdf has dropped some geometries, we need to align matrices again:
ids_geo = list(gdf.index)
idx_map = {f: i for i, f in enumerate(ids)}
keep_idx = [idx_map[f] for f in ids_geo if f in idx_map]
WP = WP[np.ix_(keep_idx, keep_idx)]
WS = WS_full.astype(float)[np.ix_(keep_idx, keep_idx)]
ids = ids_geo
N = len(ids)

# Re-extract matrix form (aligned)
# Convert wQ to full matrix aligned with ids
WQ = wQ.full()[0]

# Aligned coordinates (EPSG:3857)
gdf = gdf.to_crs(epsg=3857)
coords = np.vstack([gdf.geometry.centroid.x.values, gdf.geometry.centroid.y.values]).T

# 2.5 Build outcome & covariates (current STD outcome, 2019)
y = gdf[STD_Y_COL].values.astype(float).reshape(-1, 1)
X_raw = gdf[COVARS].values.astype(float)
X = np.hstack([np.ones((N, 1)), X_raw])   # Add intercept
K = X.shape[1]

# ============================================
# 3. SEM (multi-W) fitting: Stage-1 + Stage-2 LOSO + gate
# ============================================

# 3.1 Core helper functions --------------------------------------------------------------

def ols_fit(y, X):
    """
    Standard OLS fit for baseline:
    - returns beta, residuals, logL, AICc
    """
    N = y.shape[0]
    K = X.shape[1]
    XtX = X.T @ X
    XtX_inv = inv(XtX)
    beta = XtX_inv @ (X.T @ y)
    resid = y - X @ beta
    s2 = float((resid.T @ resid) / N)
    logL = -0.5 * N * (np.log(2 * np.pi * s2) + 1.0)
    AICc = -2 * logL + 2 * K + (2 * K * (K + 1)) / max(N - K - 1, 1)
    return beta, resid, float(logL), float(AICc)

def build_A_matrix(W_list, lambdas, mode='add'):
    """
    Build SEM A matrix = I - Σ λ_k W_k (add) or product (prod)
    """
    N = W_list[0].shape[0]
    I = np.eye(N)
    if mode == 'add':
        S = np.zeros((N, N), dtype=float)
        for Wk, lk in zip(W_list, lambdas):
            S += lk * Wk
        A = I - S
    elif mode == 'prod':
        A = I.copy()
        for Wk, lk in zip(W_list, lambdas):
            A = A @ (I - lk * Wk)
    else:
        raise ValueError("mode must be 'add' or 'prod'.")
    return A

def sem_negloglik(params, y, X, W_list, mode='add'):
    """
    SEM negative log-likelihood for optimization.
    """
    N = y.shape[0]
    K = X.shape[1]
    beta = params[:K].reshape(-1, 1)
    lambdas = params[K:-1]
    sigma2 = float(np.exp(params[-1]))  # log sigma2

    A = build_A_matrix(W_list, lambdas, mode=mode)
    e = y - X @ beta
    Ae = A @ e

    sgn, logdetA = slogdet(A)
    if sgn <= 0:
        return 1e12

    ll = logdetA - 0.5 * N * np.log(2 * np.pi * sigma2) - 0.5 * float((Ae.T @ Ae) / sigma2)
    return -ll

def sem_fit_full(y, X, W_list, mode='add', maxiter=5000):
    """
    Full SEM fit (multi-weight matrices) for Stage-1 + LOSO.
    """
    N = y.shape[0]
    K = X.shape[1]
    beta_ols, resid_ols, logL_ols, _ = ols_fit(y, X)

    # Start: OLS beta + λ=0
    init_lambdas = np.zeros(len(W_list), dtype=float)
    init_logsig2 = np.log(float((resid_ols.T @ resid_ols) / N))
    init = np.concatenate([beta_ols.flatten(), init_lambdas, [init_logsig2]])

    bounds = [(None, None)] * K + [(-0.99, 0.99)] * len(W_list) + [(None, None)]

    opt = minimize(
        sem_negloglik,
        init,
        args=(y, X, W_list, mode),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': maxiter, 'ftol': 1e-10},
    )
    if not opt.success:
        # fallback: still return best found
        pass

    params = opt.x
    beta_hat = params[:K].reshape(-1, 1)
    lambdas_hat = params[K:-1]
    sigma2_hat = float(np.exp(params[-1]))

    # log-likelihood
    logL = -sem_negloglik(params, y, X, W_list, mode=mode)

    # AICc
    p = K + len(W_list) + 1
    AICc = -2 * logL + 2 * p + (2 * p * (p + 1)) / max(N - p - 1, 1)

    # Approximate standard errors from Hessian (optional)
    se = None
    try:
        if opt.hess_inv is not None:
            Hinv = opt.hess_inv.todense() if hasattr(opt.hess_inv, "todense") else opt.hess_inv
            se = np.sqrt(np.diag(Hinv))
    except Exception:
        se = None

    return {
        'beta': beta_hat,
        'lambdas': lambdas_hat,
        'sigma2': sigma2_hat,
        'logL': float(logL),
        'AICc': float(AICc),
        'se': se,
        'opt_success': opt.success,
        'opt_message': opt.message if hasattr(opt, 'message') else "",
    }

def moran_I(resid, W_full):
    """
    Given residuals and full W (numpy array), compute Moran's I.
    """
    z = resid.flatten()
    z = z - z.mean()
    denom = np.sum(z ** 2)
    if denom <= 0:
        return np.nan
    W = W_full
    Wsum = W.sum()
    if Wsum <= 0:
        return np.nan
    num = z @ (W @ z)
    I = (len(z) / Wsum) * (num / denom)
    return float(I)

def row_standardize(W):
    """
    Row-standardize the weight matrix.
    """
    row_sum = W.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return W / row_sum

def repair_connectivity_and_rowstd(W_sub, coords_sub, k_nn=1):
    """
    If the within-state WQ submatrix has isolates, add one nearest-neighbor edge to ensure connectivity, then standardize.
    """
    W = W_sub.copy()
    row_sum = W.sum(axis=1)
    iso = np.where(row_sum == 0)[0]
    if len(iso) == 0:
        return row_standardize(W)
    for i in iso:
        d = np.sqrt(np.sum((coords_sub - coords_sub[i]) ** 2, axis=1))
        d[i] = np.inf
        j = int(np.argmin(d))
        W[i, j] = 1.0
        W[j, i] = 1.0
    return row_standardize(W)

def submatrix(M, idx):
    """
    Take submatrix M[idx, idx]
    """
    return M[np.ix_(idx, idx)]

# 3.2 Candidate weight combos: WP / WS / WQ only + WQ-related pairs only -----------------------

def candidate_models():
    """
    Single-channel: WP, WS, WQ
    Pairs: keep only WQ+WP and WQ+WS (exclude WP+WS)
    """
    models = []
    # Single-channel
    models.append(("SEM_add_1_WP", [WP]))
    models.append(("SEM_add_1_WS", [row_standardize(WS)]))
    models.append(("SEM_add_1_WQ", [row_standardize(WQ)]))
    # Two-channel: allow only WQ+WP / WQ+WS
    models.append(("SEM_add_2_WP_WQ", [WP, row_standardize(WQ)]))
    models.append(("SEM_add_2_WS_WQ", [row_standardize(WS), row_standardize(WQ)]))
    return models

# 3.3 Stage-1: US 2019 in-sample ranking --------------------------------------

beta_ols, resid_ols, logL_ols, AICc_ols = ols_fit(y, X)
mi_ols = moran_I(resid_ols, row_standardize(WQ))

stage1_rows = []
models = candidate_models()

# OLS baseline (also compute Moran's I for readability)
stage1_rows.append({
    'model': 'OLS',
    'logL': logL_ols,
    'AICc': AICc_ols,
    'absMI': abs(mi_ols),
    'MI': mi_ols,
    'lambdas': None,
})

# All single/pair SEMs
MAXITER_STAGE1 = 10000 if RUN_PROD else 6000
for name, W_list in models:
    fit = sem_fit_full(y, X, W_list, mode='add', maxiter=MAXITER_STAGE1)
    resid_sem = y - X @ fit['beta']
    A = build_A_matrix(W_list, fit['lambdas'], mode='add')
    u = A @ resid_sem
    mi = moran_I(u, row_standardize(WQ))
    stage1_rows.append({
        'model': name,
        'logL': fit['logL'],
        'AICc': fit['AICc'],
        'absMI': abs(mi),
        'MI': mi,
        'lambdas': fit['lambdas'].copy(),
    })

df_stage1 = pd.DataFrame(stage1_rows).sort_values("AICc").reset_index(drop=True)
df_stage1.to_csv(os.path.join(OUTDIR, f"stage1_insample_rank_{STD_Y_COL}.csv"), index=False)

# 3.4 Stage-2: LOSO (by STATE, for shortlist models)----------------------------

TOP_K_SINGLE = 2
TOP_K_PAIR = 5  # not many pairs; 5 is fine

# shortlist: best single + top-K overall (excluding OLS)
df_nonols = df_stage1[df_stage1['model'] != 'OLS'].copy()
shortlist_models = [df_nonols.iloc[0]['model']]
# add best single among (WP, WS, WQ)
single_names = {"SEM_add_1_WP", "SEM_add_1_WS", "SEM_add_1_WQ"}
df_single = df_nonols[df_nonols['model'].isin(list(single_names))].sort_values("AICc")
if len(df_single) > 0:
    shortlist_models.append(df_single.iloc[0]['model'])

# add top-K overall (excluding duplicates)
for m in df_nonols.head(TOP_K_PAIR + TOP_K_SINGLE)['model'].tolist():
    if m not in shortlist_models:
        shortlist_models.append(m)

shortlist_models = list(dict.fromkeys(shortlist_models))

model_to_W = {name: W_list for name, W_list in models}

def loso_one_fold(state_code, y, X, WQ_full, coords_full, ids_full):
    """
    One fold (a given STATE) leave-one-state-out:
    - fit on the training set;
    - compute logL, AICc, MoranI on the holdout state.
    """
    hold_mask = (gdf.loc[ids_full, 'STATE'].values == state_code)
    if hold_mask.sum() < 10:
        return None  # Skip very small states

    idx_hold = np.where(hold_mask)[0]
    idx_train = np.where(~hold_mask)[0]

    y_train, X_train = y[idx_train], X[idx_train]
    y_hold, X_hold = y[idx_hold], X[idx_hold]

    WQ_hold = submatrix(WQ_full, idx_hold)
    coords_hold = coords_full[idx_hold]
    WQ_hold = repair_connectivity_and_rowstd(WQ_hold, coords_hold, k_nn=1)

    out = {'STATE': state_code}

    # OLS baseline (record logL only; for sanity check)
    beta_ols_tr, resid_ols_tr, logL_ols_tr, AICc_ols_tr = ols_fit(y_train, X_train)
    out['OLS_logL_tr'] = logL_ols_tr

    for m in shortlist_models:
        W_list_full = model_to_W[m]
        W_list_tr = [submatrix(Wk, idx_train) for Wk in W_list_full]
        W_list_tr = [row_standardize(Wk) for Wk in W_list_tr]

        # Train
        fit = sem_fit_full(y_train, X_train, W_list_tr, mode='add',
                           maxiter=8000 if RUN_PROD else 4000)

        # holdout logL
        beta_hat = fit['beta']
        lambdas = fit['lambdas']

        # for holdout residual Moran, use WQ connectivity repaired within holdout
        e_hold = y_hold - X_hold @ beta_hat

        # Need A matrix for holdout with same W structure
        W_list_hold = [submatrix(Wk, idx_hold) for Wk in W_list_full]
        W_list_hold = [row_standardize(Wk) for Wk in W_list_hold]

        A_hold = build_A_matrix(W_list_hold, lambdas, mode='add')
        u_hold = A_hold @ e_hold

        mi_hold = moran_I(u_hold, WQ_hold)

        # approximate holdout log-likelihood under fitted params (pseudo)
        # Use residuals transformed by A_hold and sigma2 from fit
        sigma2 = fit['sigma2']
        sgn, logdetA = slogdet(A_hold)
        if sgn <= 0:
            logL_hold = np.nan
        else:
            logL_hold = logdetA - 0.5 * len(idx_hold) * np.log(2 * np.pi * sigma2) - 0.5 * float((u_hold.T @ u_hold) / sigma2)

        p = X.shape[1] + len(W_list_full) + 1
        AICc_hold = -2 * logL_hold + 2 * p + (2 * p * (p + 1)) / max(len(idx_hold) - p - 1, 1)

        out[f'{m}_logL'] = float(logL_hold)
        out[f'{m}_AICc'] = float(AICc_hold)
        out[f'{m}_absMI'] = abs(float(mi_hold)) if mi_hold == mi_hold else np.nan

    return out

states = sorted(gdf['STATE'].unique().tolist())

# Prepare full objects aligned to ids list
gdf_aligned = gdf.loc[ids].copy()

# Build full arrays used in LOSO
WQ_full = row_standardize(WQ)
coords_full = coords

loso_results = Parallel(n_jobs=N_JOBS)(
    delayed(loso_one_fold)(st, y, X, WQ_full, coords_full, ids) for st in states
)
loso_results = [r for r in loso_results if r is not None]
df_loso = pd.DataFrame(loso_results)
df_loso.to_csv(os.path.join(OUTDIR, f"stage2_loso_{STD_Y_COL}.csv"), index=False)

# 3.5 Summary + Pareto + Forward gate (incl. super_lenient profile)-----------------------

def loso_summary(df_loso, model_names):
    rows = []
    for m in model_names:
        AICc_vals = df_loso[f"{m}_AICc"].values
        MI_vals = df_loso[f"{m}_absMI"].values
        mean_AICc = np.nanmean(AICc_vals)
        mean_absMI = np.nanmean(MI_vals)
        rows.append({'model': m, 'mean_AICc': mean_AICc, 'mean_absMI': mean_absMI})
    return pd.DataFrame(rows).sort_values(["mean_AICc", "mean_absMI"]).reset_index(drop=True)

df_loso_sum = loso_summary(df_loso, shortlist_models)
df_loso_sum.to_csv(os.path.join(OUTDIR, f"loso_summary_{STD_Y_COL}.csv"), index=False)

# Bi-objective: smaller AICc is better; smaller absMI is better
def pareto_front(df, col1='mean_AICc', col2='mean_absMI'):
    df = df.copy()
    df['pareto'] = True
    vals = df[[col1, col2]].values
    for i in range(len(df)):
        for j in range(len(df)):
            if i == j:
                continue
            if (vals[j, 0] <= vals[i, 0]) and (vals[j, 1] <= vals[i, 1]) and ((vals[j, 0] < vals[i, 0]) or (vals[j, 1] < vals[i, 1])):
                df.loc[df.index[i], 'pareto'] = False
                break
    return df

df_pareto = pareto_front(df_loso_sum)
df_pareto.to_csv(os.path.join(OUTDIR, f"loso_pareto_{STD_Y_COL}.csv"), index=False)

# ---- Gate config: strict / lenient / super_lenient ----
GATE_PROFILE        = 'super_lenient'   # ★ US main analysis: recommended super_lenient
TIE_DELTA_REL       = 0.01             # allow 'tie-preference' when relative AICc diff ≤1%

if GATE_PROFILE == 'strict':
    # Emphasize strict CV advantage
    GATE_DAICC_REL_MAX       = -0.01   # require pair improves by at least 1%
    GATE_MI_REL_IMPROVE_MIN  = 0.00    # no required improvement; just not worse
    GATE_WINRATE_MIN         = 0.70    # AICc better in at least 70% of states
elif GATE_PROFILE == 'lenient':
    # Balance statistics and interpretability
    GATE_DAICC_REL_MAX       = 0.00    # pair mean AICc not worse than single
    GATE_MI_REL_IMPROVE_MIN  = 0.05    # at least 5% reduction in |MI|
    GATE_WINRATE_MIN         = 0.50    # better in at least half the states
else:
    # ★ Structure-leaning: prefer pair unless clearly worse
    GATE_DAICC_REL_MAX       = 0.02    # mean AICc can be at most 2% worse
    GATE_MI_REL_IMPROVE_MIN  = -0.10   # allow |MI| up to 10% worse
    GATE_WINRATE_MIN         = 0.10    # only need 10% of states better

def forward_gate(df_loso, best_single, candidate_pair):
    A_single = df_loso[f"{best_single}_AICc"].values
    A_pair   = df_loso[f"{candidate_pair}_AICc"].values
    MI_single = df_loso[f"{best_single}_absMI"].values
    MI_pair   = df_loso[f"{candidate_pair}_absMI"].values

    # ΔAICc (absolute & relative)
    mean_single = np.nanmean(A_single)
    mean_pair   = np.nanmean(A_pair)
    dAICc = mean_pair - mean_single
    dAICc_rel = dAICc / max(abs(mean_single), 1e-9)

    # Relative Moran I improvement (>0 means smaller |MI|; <0 means worse)
    mean_MI_single = np.nanmean(MI_single)
    mean_MI_pair   = np.nanmean(MI_pair)
    mi_rel_improve = (mean_MI_single - mean_MI_pair) / max(mean_MI_single, 1e-9)

    # State-level win_rate: share of states where pair AICc is smaller
    win_rate = np.nanmean((A_pair < A_single).astype(float))

    # Gate decision
    ok = (dAICc_rel <= GATE_DAICC_REL_MAX) and (mi_rel_improve >= GATE_MI_REL_IMPROVE_MIN) and (win_rate >= GATE_WINRATE_MIN)

    # If gate fails but ΔAICc_rel is very small (near tie), still may prefer pair
    tie_pref = (dAICc_rel <= TIE_DELTA_REL)

    return {
        'mean_single': mean_single,
        'mean_pair': mean_pair,
        'dAICc': dAICc,
        'dAICc_rel': dAICc_rel,
        'mean_MI_single': mean_MI_single,
        'mean_MI_pair': mean_MI_pair,
        'mi_rel_improve': mi_rel_improve,
        'win_rate': win_rate,
        'gate_ok': bool(ok),
        'tie_pref': bool(tie_pref),
    }

# Identify best single (by LOSO mean_AICc among singles)
df_single_loso = df_loso_sum[df_loso_sum['model'].isin(list(single_names))].copy()
best_single = df_single_loso.sort_values('mean_AICc').iloc[0]['model']

# Candidate pairs among shortlist
pair_names = [m for m in shortlist_models if m in ("SEM_add_2_WP_WQ", "SEM_add_2_WS_WQ")]
gate_records = []
for p in pair_names:
    rec = forward_gate(df_loso, best_single, p)
    rec['best_single'] = best_single
    rec['pair'] = p
    gate_records.append(rec)
df_gate = pd.DataFrame(gate_records)
df_gate.to_csv(os.path.join(OUTDIR, f"gate_check_{STD_Y_COL}.csv"), index=False)

# Final model selection
final_model_main = best_single
if len(df_gate) > 0:
    # prefer pair that passes gate; if multiple, choose the one with smaller mean_pair
    df_ok = df_gate[df_gate['gate_ok'] == True].copy()
    if len(df_ok) > 0:
        best_pair = df_ok.sort_values('mean_pair').iloc[0]['pair']
        final_model_main = best_pair
    else:
        # tie preference
        df_tie = df_gate[df_gate['tie_pref'] == True].copy()
        if len(df_tie) > 0:
            best_pair = df_tie.sort_values('mean_pair').iloc[0]['pair']
            final_model_main = best_pair

print("[Final] best_single =", best_single, "final_model_main =", final_model_main)

# 3.6 Channel importance (LOSO wAICc weights)----------------------------------

def wAICc_weights(AICc_array):
    """
    Compute wAICc weights from AICc.
    """
    a = np.array(AICc_array, dtype=float)
    amin = np.nanmin(a)
    delta = a - amin
    w = np.exp(-0.5 * delta)
    w = w / np.nansum(w)
    return w

def channel_importance(df_loso, model_names):
    """
    Based on wAICc weights across all candidate SEM models in LOSO,
    summarize the mean importance of each channel.
    """
    # Compute mean AICc per model across states
    mean_AICc = np.array([np.nanmean(df_loso[f"{m}_AICc"].values) for m in model_names])
    w = wAICc_weights(mean_AICc)
    imp = {'WP': 0.0, 'WS': 0.0, 'WQ': 0.0}
    for mi, m in enumerate(model_names):
        if "WP" in m:
            imp['WP'] += w[mi]
        if "WS" in m:
            imp['WS'] += w[mi]
        if "WQ" in m:
            imp['WQ'] += w[mi]
    return imp

imp = channel_importance(df_loso, shortlist_models)
pd.Series(imp).to_csv(os.path.join(OUTDIR, f"channel_importance_{STD_Y_COL}.csv"))

# 3.7 Save main-analysis results --------------------------------------------------------

# Hero model: follow the same logic as the DeepSouth version
hero_model = final_model_main

# ============================================
# 4. Spillover classification (hero vs WQ-only baseline)
# ============================================

#    —— automatically handles hero = single-channel / two-channel
# 4.1 Parse W_list from model name (auto-detect WP / WS / WQ)
def parse_W_list(model_name):
    """
    For example:
      'SEM_add_1_WQ' -> ['WQ']
      'SEM_add_2_WP_WQ' -> ['WP', 'WQ']
    """
    parts = model_name.split("_")
    # find the channels after the count (e.g. ..._1_WQ or ..._2_WP_WQ)
    chs = []
    for p in parts:
        if p in ("WP", "WS", "WQ"):
            chs.append(p)
    return chs

hero_channels = parse_W_list(hero_model)
# hero_model has already been selected by the gate above

# 4.2 Fit hero model & WQ-only model (full data)
# baseline: WQ-only; if no WQ (rare), fall back to OLS linear prediction
def get_W_list_from_channels(channels):
    W_list = []
    for ch in channels:
        if ch == 'WP':
            W_list.append(WP)
        elif ch == 'WS':
            W_list.append(row_standardize(WS))
        elif ch == 'WQ':
            W_list.append(row_standardize(WQ))
    return W_list

W_list_hero = get_W_list_from_channels(hero_channels)

fit_hero = sem_fit_full(y, X, W_list_hero, mode='add', maxiter=12000 if RUN_PROD else 7000)

# baseline WQ-only
fit_wq = None
if 'WQ' in hero_channels or True:
    W_list_wq = [row_standardize(WQ)]
    fit_wq = sem_fit_full(y, X, W_list_wq, mode='add', maxiter=12000 if RUN_PROD else 7000)

def sem_fitted_values(y, X, fit, W_list):
    """
    Compute fitted values for SEM:
    """
    beta_hat = fit['beta']
    return (X @ beta_hat)

# hero fitted values (single or two-channel handled uniformly)
yhat_hero = sem_fitted_values(y, X, fit_hero, W_list_hero)

# baseline fitted values: prefer WQ-only SEM, otherwise OLS linear prediction
if fit_wq is not None:
    yhat_wq = sem_fitted_values(y, X, fit_wq, [row_standardize(WQ)])
else:
    beta_ols_all, _, _, _ = ols_fit(y, X)
    yhat_wq = X @ beta_ols_all

# 4.3 Compute rank shift (hero vs WQ baseline)
rank_hero = pd.Series(yhat_hero.flatten(), index=ids).rank(ascending=False, method='average')
rank_wq   = pd.Series(yhat_wq.flatten(),   index=ids).rank(ascending=False, method='average')
delta_rank = rank_hero - rank_wq

# 4.4 Build spill_type: Q = WQ, B = 'bridging channels' in hero excluding WQ

# Q strength
Q_strength = pd.Series(row_standardize(WQ).sum(axis=1), index=ids)

# bridging channels = hero_channels excluding WQ (may be [WP], [WS], [WP,WS], or empty)
bridging_channels = [ch for ch in hero_channels if ch != 'WQ']

if len(bridging_channels) == 0:
    B_strength = pd.Series(np.zeros(N), index=ids)
else:
    Bmats = []
    for ch in bridging_channels:
        if ch == 'WP':
            Bmats.append(WP)
        elif ch == 'WS':
            Bmats.append(row_standardize(WS))
    # Sum all bridging channels into one 'composite B'
    B_sum = np.zeros((N, N), dtype=float)
    for Bm in Bmats:
        B_sum += Bm
    B_strength = pd.Series(B_sum.sum(axis=1), index=ids)

# Split High / Low by median
Q_med = Q_strength.median()
B_med = B_strength.median()

is_highQ = (Q_strength >= Q_med)
is_highB = (B_strength >= B_med)

# Generalized 'bridging': LowQ & HighB (regardless of whether B is WP, WS, or both)
spill_type = pd.Series(index=ids, dtype=object)
spill_type[(is_highQ) & (is_highB)] = "HighQ_HighB"
spill_type[(is_highQ) & (~is_highB)] = "HighQ_LowB"
spill_type[(~is_highQ) & (is_highB)] = "LowQ_HighB"
spill_type[(~is_highQ) & (~is_highB)] = "LowQ_LowB"

# Record which bridging channels were used
bridging_label = "+".join(bridging_channels) if len(bridging_channels) > 0 else "OnlyQ"

# hero has only WQ (e.g., SEM_add_1_WQ)
if len(bridging_channels) == 0:
    spill_type[:] = "OnlyQ"

# 4.5 Write shapefile + CSV (US prefix + outcome_tag)

out_gdf = gdf_aligned.copy()
out_gdf['delta_rank'] = delta_rank.loc[out_gdf.index].values
out_gdf['spill_type'] = spill_type.loc[out_gdf.index].values
out_gdf['is_bridging'] = ((~is_highQ) & (is_highB)).loc[out_gdf.index].values
out_gdf['bridging_ch'] = bridging_label

out_shp = os.path.join(OUTDIR, f"US_spillover_{STD_Y_COL}.shp")
out_csv = os.path.join(OUTDIR, f"US_spillover_{STD_Y_COL}.csv")
out_gdf.to_file(out_shp)
out_gdf.drop(columns='geometry').to_csv(out_csv, index=True)

# ============================================
# 5. Rewiring null test (degree-preserving row-wise rewiring)
# ============================================

B_NULL = 200 if RUN_PROD else 40
MAXITER_NULL = 4000 if RUN_PROD else 2500

# 5.1 Choose the 'target channel' to rewire:
#     Priority: bridging channels in hero excluding WQ; if none, can only rewire WQ/single-channel
if len(bridging_channels) > 0:
    target_ch = bridging_channels[0]  # pick first
else:
    target_ch = "WQ"

# Original hero model AICc and |Moran I|
hero_AICc = fit_hero['AICc']
hero_resid = y - X @ fit_hero['beta']
A_hero = build_A_matrix(W_list_hero, fit_hero['lambdas'], mode='add')
u_hero = A_hero @ hero_resid
hero_MI = moran_I(u_hero, row_standardize(WQ))

# 5.2 Fast SEM fit (for nulls)
def sem_fit_fast(y, X, W_list, maxiter=2500):
    return sem_fit_full(y, X, W_list, mode='add', maxiter=maxiter)

# 5.3 Rewiring: shuffle nonzero weights within each row (degree-preserving)
def rowwise_rewire(W):
    """
    Degree-preserving rewiring for a given row-standardized matrix:
      - within each row, permute weights among nonzero positions only
      - keep island rows (all zeros) unchanged
    """
    W2 = W.copy()
    for i in range(W2.shape[0]):
        nz = np.where(W2[i, :] > 0)[0]
        if len(nz) <= 1:
            continue
        vals = W2[i, nz].copy()
        rng_global.shuffle(vals)
        W2[i, nz] = vals
    return W2

# 5.4 Generate rewired nulls and fit the 'same-structure hero'
def hero_fit_with_rewired(WP_in, WS_in, WQ_in, hero_channels, maxiter=3000):
    W_list = []
    for ch in hero_channels:
        if ch == "WP":
            W_list.append(WP_in)
        elif ch == "WS":
            W_list.append(WS_in)
        elif ch == "WQ":
            W_list.append(WQ_in)
    fit = sem_fit_fast(y, X, W_list, maxiter=maxiter)
    resid = y - X @ fit['beta']
    A = build_A_matrix(W_list, fit['lambdas'], mode='add')
    u = A @ resid
    mi = moran_I(u, row_standardize(WQ_in))
    return fit['AICc'], abs(mi)

null_AICc = []
null_absMI = []

for b in range(B_NULL):
    WP_b = WP.copy()
    WS_b = row_standardize(WS).copy()
    WQ_b = row_standardize(WQ).copy()

    if target_ch == "WP":
        WP_b = rowwise_rewire(WP_b)
    elif target_ch == "WS":
        WS_b = rowwise_rewire(WS_b)
    else:
        WQ_b = rowwise_rewire(WQ_b)

    # Row-wise rewiring on target_ch; keep other channels unchanged
    # Use the channel combination of the 'same hero_model' name (but one channel has been rewired)
    AICc_b, absMI_b = hero_fit_with_rewired(WP_b, WS_b, WQ_b, hero_channels, maxiter=MAXITER_NULL)
    null_AICc.append(AICc_b)
    null_absMI.append(absMI_b)

df_null = pd.DataFrame({'AICc_null': null_AICc, 'absMI_null': null_absMI})
df_null.to_csv(os.path.join(OUTDIR, f"rewiring_null_{STD_Y_COL}.csv"), index=False)

# ============================================
# 6. Perturbation tests (WP/WS only; keep WQ fixed)
# ============================================

#    —— same logic as DeepSouth: perturb WP/WS only; keep Q fixed
B_PERT = 200 if RUN_PROD else 40

# 6.1 Pre-construct Moran object for WQ (no perturbation)
WQ_base = row_standardize(WQ)

# 6.2 Fast SEM (lower optimization precision is fine)
MAXITER_PERT = 4500 if RUN_PROD else 2500

# 6.3 FAST Stage-1: only check WQ-only / WP+WQ / WS+WQ
fast_candidates = []
fast_candidates.append(("SEM_add_1_WQ", ["WQ"]))
if "SEM_add_2_WP_WQ" in model_to_W:
    # Add WP+WQ / WS+WQ if present
    fast_candidates.append(("SEM_add_2_WP_WQ", ["WP", "WQ"]))
if "SEM_add_2_WS_WQ" in model_to_W:
    fast_candidates.append(("SEM_add_2_WS_WQ", ["WS", "WQ"]))

def perturb_matrix(W, drop_prob=0.02, mult_noise=0.05):
    Wp = W.copy()
    # Randomly drop edges (symmetric)
    mask = rng_global.random(Wp.shape) < drop_prob
    Wp[mask] = 0.0
    # Multiplicative noise on edge weights
    noise = 1.0 + mult_noise * rng_global.normal(size=Wp.shape)
    Wp = Wp * noise
    Wp = (Wp + Wp.T) / 2.0
    Wp[Wp < 0] = 0.0
    Wp = row_standardize(Wp)  # Row-standardize
    return Wp

# 6.5 Main loop
pert_records = []
for b in range(B_PERT):
    WP_b = perturb_matrix(WP, drop_prob=0.02, mult_noise=0.05)
    WS_b = perturb_matrix(row_standardize(WS), drop_prob=0.02, mult_noise=0.05)
    WQ_b = WQ_base

    for name, chs in fast_candidates:
        W_list = []
        for ch in chs:
            if ch == "WP":
                W_list.append(WP_b)
            elif ch == "WS":
                W_list.append(WS_b)
            else:
                W_list.append(WQ_b)

        fit = sem_fit_fast(y, X, W_list, maxiter=MAXITER_PERT)
        resid = y - X @ fit['beta']
        A = build_A_matrix(W_list, fit['lambdas'], mode='add')
        u = A @ resid
        mi = moran_I(u, WQ_b)
        pert_records.append({
            'b': b,
            'model': name,
            'AICc': fit['AICc'],
            'absMI': abs(mi),
        })

df_pert = pd.DataFrame(pert_records)
df_pert.to_csv(os.path.join(OUTDIR, f"perturb_tests_{STD_Y_COL}.csv"), index=False)

# Model frequency
# Channel inclusion frequency
# (These can be computed downstream from outputs)

# ============================================
# 7. Temporal robustness: apply 2019 hero structure to 2018 outcomes
# ============================================

#    —— hero structure is automatically reused, adapted to current outcome
# 7.1 Parse hero channel structure (reuse the structure of 2019 final_model_main)
hero_chs = parse_W_list(final_model_main)

# 7.2 Intersection sample: complete 2018 & 2019 outcomes + covariates
y2018_col = TEMPORAL_2018_COL.get(STD_Y_COL, None)
if y2018_col is None or y2018_col not in gdf_aligned.columns:
    print("[Temporal] 2018 column not available; skip temporal robustness.")
else:
    gdf_tmp = gdf_aligned.copy()
    need_cols_tmp = [STD_Y_COL, y2018_col] + COVARS
    gdf_tmp = gdf_tmp.dropna(subset=need_cols_tmp).copy()
    ids_tmp = list(gdf_tmp.index)
    idx_map2 = {f: i for i, f in enumerate(ids)}
    keep_idx2 = [idx_map2[f] for f in ids_tmp]

    y2019_t = gdf_tmp[STD_Y_COL].values.astype(float).reshape(-1, 1)
    y2018_t = gdf_tmp[y2018_col].values.astype(float).reshape(-1, 1)
    X_t = np.hstack([np.ones((len(ids_tmp), 1)), gdf_tmp[COVARS].values.astype(float)])

    # W_list for hero on the intersection sample
    def slice_W_for_ids(W, keep_idx):
        return W[np.ix_(keep_idx, keep_idx)]

    WP_t = slice_W_for_ids(WP, keep_idx2)
    WS_t = slice_W_for_ids(row_standardize(WS), keep_idx2)
    WQ_t = slice_W_for_ids(row_standardize(WQ), keep_idx2)

    W_list_t = []
    for ch in hero_chs:
        if ch == "WP":
            W_list_t.append(row_standardize(WP_t))
        elif ch == "WS":
            W_list_t.append(row_standardize(WS_t))
        else:
            W_list_t.append(row_standardize(WQ_t))

    # Fit for 2019 and 2018 with same hero structure but re-estimate parameters
    fit_2019 = sem_fit_full(y2019_t, X_t, W_list_t, mode='add', maxiter=12000 if RUN_PROD else 7000)
    fit_2018 = sem_fit_full(y2018_t, X_t, W_list_t, mode='add', maxiter=12000 if RUN_PROD else 7000)

    # 3) Hero SEM (reuse 2019 channel structure)
    temporal_out = pd.DataFrame({
        'year': [2019, 2018],
        'logL': [fit_2019['logL'], fit_2018['logL']],
        'AICc': [fit_2019['AICc'], fit_2018['AICc']],
        'lambdas': [",".join([f"{x:.6f}" for x in fit_2019['lambdas']]),
                    ",".join([f"{x:.6f}" for x in fit_2018['lambdas']])],
    })

    # 7.4 Interannual comparison of hero λ (also exports hero λ)
    temporal_out.to_csv(os.path.join(OUTDIR, f"temporal_hero_{STD_Y_COL}.csv"), index=False)

    # 7.5 Save results (filename includes outcome_tag)
    print("[Temporal] saved temporal_hero CSV.")

# ============================================
# 8. Mapping figures: basemap + spillover types + delta_rank
# ============================================

#    —— Unified basemap: boundaries + missing-data background + standard legend
plt.rcParams['figure.facecolor'] = "white"  # Set white background

# 8.1 Basemap: US counties + missing flag (based on current y_col)
def build_us_basemap(shp_path, y_col):
    """
    Read US county shapefile and build a unified basemap:
    - clean FIPS
    - drop missing geometry
    - has_data: whether the current outcome is missing
    - project to an appropriate UTM CRS
    """
    g0 = gpd.read_file(shp_path)
    if 'FIPS' not in g0.columns:
        raise ValueError("FIPS column not found.")
    g0['FIPS'] = g0['FIPS'].astype(str).str.zfill(5)
    g0 = g0.dropna(subset=['geometry']).copy()
    if y_col in g0.columns:
        g0['has_data'] = ~g0[y_col].isna()
    else:
        g0['has_data'] = True

    # Choose UTM zone based on overall centroid longitude (assume original CRS is lon/lat)
    g0_ll = g0.to_crs(epsg=4326)
    lon = g0_ll.geometry.centroid.x.mean()
    utm_zone = int((lon + 180) / 6) + 1
    utm_crs = f"EPSG:326{utm_zone:02d}"   # Northern hemisphere UTM
    g0 = g0.to_crs(utm_crs)
    return g0

# Use shp_path (read above) and current y_col
g_base = build_us_basemap(shp_path, STD_Y_COL)

# Spillover shapefile for the model sample
g_spill = gpd.read_file(out_shp).to_crs(g_base.crs)

# Counties actually used by the model (to distinguish background: has outcome but excluded)
model_ids = set(g_spill['FIPS'].astype(str).str.zfill(5).tolist())
g_base['in_model'] = g_base['FIPS'].astype(str).str.zfill(5).apply(lambda x: x in model_ids)

# Background colors
missing_color   = "#4f4f4f"  # dark gray: current outcome missing
nonmodel_color  = "#d9d9d9"  # light gray: has outcome but not in SEM model

def draw_basemap(ax, g_base):
    """
    Draw the unified basemap in the given CRS:
    - thick black boundary
    - missing counties (has_data=False) in dark gray
    - counties with outcome but not in model sample in light gray
    """
    # Outer boundary
    g_base.boundary.plot(ax=ax, linewidth=0.2, edgecolor="black")

    # Counties missing the outcome
    g_base.loc[~g_base['has_data']].plot(ax=ax, color=missing_color, linewidth=0)

    # Counties with outcome but not in model sample
    g_base.loc[(g_base['has_data']) & (~g_base['in_model'])].plot(ax=ax, color=nonmodel_color, linewidth=0)

# 8.2 spillover type column + colors (dynamically supports OnlyQ)
if 'spill_type' not in g_spill.columns:
    raise ValueError("Cannot find spill_type column; please check shp field name.")

# Backward-compatible naming (...P → ...B)
g_spill['spill_type'] = g_spill['spill_type'].replace({
    "HighQ_HighP": "HighQ_HighB",
    "HighQ_LowP": "HighQ_LowB",
    "LowQ_HighP": "LowQ_HighB",
    "LowQ_LowP": "LowQ_LowB",
})

if (g_spill['spill_type'] == "OnlyQ").any():
    # Case 1: hero has only WQ → spill_type = 'OnlyQ'
    color_map = {"OnlyQ": "#3182bd"}  # blue
else:
    # Case 2: four Q×B combinations
    color_map = {
        "HighQ_HighB": "#d73027",  # deep red
        "HighQ_LowB":  "#fc8d59",  # light red
        "LowQ_HighB":  "#4575b4",  # blue
        "LowQ_LowB":   "#fee090",  # light yellow; distinct from light-gray background
    }

# bridging channel label (for title)
title_bridge = bridging_label

# 8.3 Figure A: four spillover types + missing/background legend
figA, axA = plt.subplots(1, 1, figsize=(12, 7))

# Draw the unified basemap first
draw_basemap(axA, g_base)

# Then overlay spillover categories
for k, c in color_map.items():
    g_spill.loc[g_spill['spill_type'] == k].plot(ax=axA, color=c, linewidth=0)

axA.set_axis_off()
axA.set_title(f"Spillover types ({STD_Y_COL}) | Hero={hero_model} | Bridging={title_bridge}")

# Legend: spillover types + missing/not-modeled
import matplotlib.patches as mpatches
handles = []
for k, c in color_map.items():
    handles.append(mpatches.Patch(color=c, label=k))
handles.append(mpatches.Patch(color=missing_color, label="Missing outcome"))
handles.append(mpatches.Patch(color=nonmodel_color, label="Has outcome, not in model"))
axA.legend(handles=handles, loc="lower left", frameon=True, fontsize=9)

figA.tight_layout()
figA.savefig(os.path.join(OUTDIR, f"FigA_spillover_{STD_Y_COL}.png"), dpi=200)

# Background explanation
# 8.4 Prepare data for rank plots: is_bridging / delta_rank

# bridging flag column (compatible with is_bridgin)
if 'is_bridging' not in g_spill.columns and 'is_bridgin' in g_spill.columns:
    g_spill['is_bridging'] = g_spill['is_bridgin']

if 'is_bridging' not in g_spill.columns:
    raise ValueError("Cannot find is_bridging / is_bridgin field; please check shp.")

# delta_rank column
if 'delta_rank' not in g_spill.columns:
    raise ValueError("Cannot find delta_rank field; please check shp.")

# 8.5 Figure B1: Δrank for all bridging counties (symmetric color scale) + background
figB1, axB1 = plt.subplots(1, 1, figsize=(12, 7))

# Basemap
draw_basemap(axB1, g_base)

# All bridging counties
g_bridge = g_spill[g_spill['is_bridging'] == True].copy()
if len(g_bridge) == 0:
    print("Warning: no counties with is_bridge == True; please check the field.")
else:
    vmax = np.nanmax(np.abs(g_bridge['delta_rank'].values))
    g_bridge.plot(ax=axB1,
                  column='delta_rank',
                  cmap='coolwarm',
                  linewidth=0,
                  legend=True,
                  vmin=-vmax, vmax=vmax,   # ★ symmetric color scale
                  missing_kwds={"color": missing_color})
axB1.set_axis_off()
axB1.set_title(f"Δrank (Hero - WQ) for bridging counties | {STD_Y_COL}")

# Small legend: label background only (avoid mixing with the colorbar)
handles_bg = [
    mpatches.Patch(color=missing_color, label="Missing outcome"),
    mpatches.Patch(color=nonmodel_color, label="Has outcome, not in model"),
]
axB1.legend(handles=handles_bg, loc="lower left", frameon=True, fontsize=9)

figB1.tight_layout()
figB1.savefig(os.path.join(OUTDIR, f"FigB1_delta_rank_bridging_{STD_Y_COL}.png"), dpi=200)

# 8.6 Figure B2: 'typical bridging counties' with Δrank ≥ THRESH + background
THRESH = 500
g_top_bridge = g_bridge[g_bridge['delta_rank'] >= THRESH].copy()
print(f"Number of bridging counties with Δrank ≥ {THRESH} (US):", len(g_top_bridge))

figB2, axB2 = plt.subplots(1, 1, figsize=(12, 7))

# Basemap
draw_basemap(axB2, g_base)

# All bridging counties in light blue
g_bridge.plot(ax=axB2, color="#9ecae1", linewidth=0)

# High-Δrank focus counties in a red gradient
if len(g_top_bridge) > 0:
    g_top_bridge.plot(ax=axB2, color="#de2d26", linewidth=0)

axB2.set_axis_off()
axB2.set_title(f"Typical bridging counties (Δrank ≥ {THRESH}) | {STD_Y_COL}")

# Background legend
handles_bg2 = [
    mpatches.Patch(color=missing_color, label="Missing outcome"),
    mpatches.Patch(color=nonmodel_color, label="Has outcome, not in model"),
    mpatches.Patch(color="#9ecae1", label="Bridging counties"),
    mpatches.Patch(color="#de2d26", label=f"Δrank ≥ {THRESH}"),
]
axB2.legend(handles=handles_bg2, loc="lower left", frameon=True, fontsize=9)

figB2.tight_layout()
figB2.savefig(os.path.join(OUTDIR, f"FigB2_top_bridging_{STD_Y_COL}.png"), dpi=200)

print("[Done] US main analysis pipeline complete.")

###Add or prod

In [ ]:
# ============================================
# 0. Imports & Global configs
# ============================================
import numpy as np
import pandas as pd
import geopandas as gpd

from numpy.linalg import slogdet
from scipy.optimize import minimize

from esda.moran import Moran
from libpysal.weights import util, W, Queen
from joblib import Parallel, delayed

np.seterr(all='ignore')

RANDOM_SEED = 7
rng_global = np.random.default_rng(RANDOM_SEED)

# Parallel configuration
N_JOBS = -1   # -1 uses all cores; set to 1 if unstable

# ------------- Outcome config: switch STD outcome with one line ----------------
# Options: 'PercenHIVp', 'GonorrheaR', 'ChlamydiaR'
y_col = 'PercenHIVp'

OUTCOME_TAG = {
    'PercenHIVp': 'HIV',
    'GonorrheaR': 'Gonorrhea',
    'ChlamydiaR': 'Chlamydia'
}
outcome_tag = OUTCOME_TAG.get(y_col, y_col)

# Temporal robustness mapping (we only need outcome_tag here; no need to actually run 2018)
TEMPORAL_2018_COL = {
    'PercenHIVp': 'RateHIV2018',
    'GonorrheaR': 'RateGonorrhea2018',
    'ChlamydiaR': 'RateChlamydia2018'
}
outcome_2018_col = TEMPORAL_2018_COL.get(y_col, None)

# ============================================
# 1. Read US shapefile & basic preprocessing
# ============================================

shp_path = "US_HIV_Merged_total_final.shp"
gdf = gpd.read_file(shp_path)

if 'FIPS' not in gdf.columns:
    raise KeyError("Shapefile needs a 'FIPS' field.")

# Strip leading zeros from FIPS and drop duplicates
gdf['FIPS'] = gdf['FIPS'].astype(str).str.lstrip('0')
gdf = gdf.drop_duplicates(subset=['FIPS'])

# Fix potentially truncated 2018 field names
rename_map = {
    'RateChlamy': 'RateChlamydia2018',
    'RateGonorr': 'RateGonorrhea2018',
    'RateHIV201': 'RateHIV2018'
}
for short, long_name in rename_map.items():
    if (short in gdf.columns) and (long_name not in gdf.columns):
        gdf = gdf.rename(columns={short: long_name})

# Covariates
X_cols = [
    'Population','UrbanRural','Female','Old','Black',
    'Noinsuranc','Poverty','crime16to1','Dissimilar'
]

cols_keep = [
    'FIPS',
    'GonorrheaR','ChlamydiaR','PercenHIVp',
    *X_cols,
    'RateChlamydia2018','RateGonorrhea2018','RateHIV2018',
    'geometry'
]

df_geo = gdf[cols_keep].copy()

# Drop missing values for the current 2019 outcome + covariates (coarse filter first)
df_geo = df_geo.dropna(subset=[y_col] + X_cols)

# State code
def fips2state(x):
    return str(x).zfill(5)[:2]

df_geo['STATE'] = df_geo['FIPS'].apply(fips2state)
df_geo = gpd.GeoDataFrame(df_geo, geometry='geometry', crs=gdf.crs)

# ============================================
# 2. Build spatial weights & design matrix (US)
# ============================================

# 2.1 WP：Twitter PCI mobility
pci_path = "US_County_PCI_2019.csv"
pci = pd.read_csv(pci_path)

pci['place_i'] = pci['place_i'].astype(str).str.lstrip('0')
pci['place_j'] = pci['place_j'].astype(str).str.lstrip('0')

pci_f = pci[
    pci['place_i'].isin(df_geo['FIPS']) &
    pci['place_j'].isin(df_geo['FIPS'])
].copy()

A_pci = pci_f.pivot_table(
    index='place_i', columns='place_j',
    values='pci', fill_value=0
)
np.fill_diagonal(A_pci.values, 0.0)
A = A_pci.values.astype(float)
A = A + A.T - np.diag(np.diag(A))  # Symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP = A / row_sum
wP = util.full2W(WP, ids=list(A_pci.index))

# 2.2 WS：Facebook SCI social ties
sci_path = "processed_sci_summary.csv"
sci = pd.read_csv(sci_path)

sci['user_loc'] = sci['user_loc'].astype(str).str.lstrip('0')
sci['tfr_loc']  = sci['tfr_loc'].astype(str).str.lstrip('0')

sci_f = sci[
    sci['user_loc'].isin(df_geo['FIPS']) &
    sci['tfr_loc'].isin(df_geo['FIPS'])
].copy()

A_sci = sci_f.pivot_table(
    index='user_loc', columns='tfr_loc',
    values='tscaled_sci', fill_value=0
)
np.fill_diagonal(A_sci.values, 0.0)
B = A_sci.values.astype(float)
B = B + B.T - np.diag(np.diag(B))
row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WS = B / row_sum
wS = util.full2W(WS, ids=list(A_sci.index))

# 2.3 WQ: Queen contiguity (US-wide)
g3857 = df_geo.to_crs(epsg=3857)
wQ = Queen.from_dataframe(g3857, ids=list(df_geo['FIPS']))
wQ.transform = 'R'
WQ, _ = wQ.full()

# 2.4 Align common ID set & order (WP / WS / WQ / df_geo unified)

common = (
    set(df_geo['FIPS']) &
    set(wP.id_order) &
    set(wS.id_order) &
    set(wQ.id_order)
)

def subset_W(w: W, keep_ids):
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))
    keep = set(keep_ids)
    new_neighbors, new_weights = {}, {}
    for i in w.id_order:
        if i not in keep:
            continue
        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])
        kn, kw = [], []
        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kn.append(nb); kw.append(wt)
        new_neighbors[i] = kn
        new_weights[i] = kw
    return W(new_neighbors, new_weights, ids=keep_ids)

df_geo = df_geo[df_geo['FIPS'].isin(common)].copy()
wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

df_geo = df_geo.set_index('FIPS').loc[wQ.id_order].reset_index()
IDs   = list(wQ.id_order)
STATE = df_geo['STATE'].values

# Convert to matrix form again (after alignment)
WP, _ = wP.full()
WS, _ = wS.full()
WQ, _ = wQ.full()

# Aligned coordinates (EPSG:3857)
XY_all = np.vstack([
    g3857.set_index('FIPS').loc[IDs].geometry.centroid.x.values,
    g3857.set_index('FIPS').loc[IDs].geometry.centroid.y.values
]).T

# 2.5 Build outcome & covariates (current STD outcome, 2019)
df = df_geo[['FIPS', y_col] + X_cols + ['STATE','geometry']].dropna().copy()

y = df[y_col].values
X_raw = df[X_cols].values
N = X_raw.shape[0]
X = np.hstack([np.ones((N, 1)), X_raw])   # Add intercept
K = X.shape[1]

# ============================================
# 3. SEM core helpers + Stage-1 & Stage-2 + Gate
# ============================================

# 3.1 Core helper functions --------------------------------------------------------------

def fit_ols(y, X):
    """
    Standard OLS fit for the baseline:
    - returns beta, residuals, logL, AICc
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)
    sse = float(e.T.dot(e))
    N_, K_ = X.shape
    df_ = N_ - K_
    sigma2 = sse / df_
    logL = -(N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    AIC  = -2*logL + 2*K_
    AICc = (AIC + (2*K_*(K_+1)) / (N_-K_-1)
            if (N_-K_-1) > 0 else np.nan)
    return {
        'model': 'OLS',
        'beta': beta,
        'logL': logL,
        'AICc': AICc,
        'resid': e
    }


def build_A_and_logdet(lam, W_list, combo='add'):
    """
    Build the SEM A matrix = I - Σ λ_k W_k (add) or a matrix product chain (prod)
    """
    N_ = W_list[0].shape[0]
    I = np.eye(N_)

    if combo == 'add':
        A = I.copy()
        for i, Wk in enumerate(W_list):
            A -= lam[i] * Wk
        sign, logdetA = slogdet(A)
        if sign <= 0:
            return None, None
        return A, logdetA

    elif combo == 'prod':
        A = I.copy()
        logdet_sum = 0.0
        for i, Wk in enumerate(W_list):
            Mi = I - lam[i] * Wk
            sign_i, logdet_i = slogdet(Mi)
            if sign_i <= 0:
                return None, None
            logdet_sum += logdet_i
            A = Mi @ A
        return A, logdet_sum

    else:
        raise ValueError("combo must be 'add' or 'prod'")


def neglog_sem(params, y, X, W_list, combo='add'):
    """
    SEM negative log-likelihood (for optimization).
    params = [λ_1,...,λ_m, β_0,...,β_{K-1}]
    """
    N_, K_ = X.shape
    m = len(W_list)
    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list, combo=combo)
    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)
    sse = float(v.T.dot(v))

    df_ = N_ - (K_ + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = logdetA - (N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    return -logL


def fit_sem_multi(y, X, W_list, method='L-BFGS-B', combo='add'):
    """
    Full SEM fit (multi-weight-matrix), used for Stage-1 / LOSO / Add vs Prod。
    """
    N_, K_ = X.shape
    m = len(W_list)

    # Initial values: OLS beta + lambda=0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)]*m + [(-np.inf, np.inf)]*K_

    res = minimize(
        neglog_sem, x0,
        args=(y, X, W_list, combo),
        method=method,
        bounds=bounds,
        options={'maxiter': 200, 'ftol': 1e-4, 'gtol': 1e-4}
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K_
    AIC  = -2*logL + 2*p
    AICc = AIC + (2*p*(p+1))/(N_-p-1) if (N_-p-1) > 0 else np.nan

    A, _ = build_A_and_logdet(lam, W_list, combo=combo)
    v = A.dot(y - X.dot(beta))

    # Hessian-based approximate standard errors (optional)
    if hasattr(res, 'hess_inv'):
        try:
            Hin = res.hess_inv.todense()
        except Exception:
            Hin = np.asarray(res.hess_inv)
        se = np.sqrt(np.diag(Hin)) if Hin.shape == (p, p) else np.full(p, np.nan)
    else:
        se = np.full(p, np.nan)

    return {
        'model': f'SEM_{combo}_{len(W_list)}W',
        'lam': lam,
        'beta': beta,
        'logL': logL,
        'AICc': AICc,
        'resid': v,
        'combo': combo,
        'success': res.success,
        'se': se
    }


def moran_I_val_from_array(resid, W_array, ids):
    """
    Given residuals and full W (numpy array), compute Moran's I.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)
    return mi.I, mi.p_norm


def row_standardize(A):
    """
    Row-standardize the weight matrix.
    """
    R = A.astype(float)
    rs = R.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    return R / rs


def ensure_connected_within_state(W_sub, XY_sub):
    """
    If a within-state WQ submatrix has isolates, connect each isolate once to its nearest neighbor, then re-standardize.
    """
    R = W_sub.copy()
    deg = R.sum(axis=1)

    if np.any(deg == 0):
        D = np.linalg.norm(
            XY_sub[:, None, :] - XY_sub[None, :, :], axis=2
        )
        np.fill_diagonal(D, np.inf)
        lonely = np.where(deg == 0)[0]
        for i in lonely:
            j = int(D[i].argmin())
            if np.isfinite(D[i, j]):
                R[i, j] = 1.0
                R[j, i] = 1.0
        R = row_standardize(R)

    return R


def submatrix(M, idx):
    """
    Take a submatrix M[idx, idx]
    """
    return M[np.ix_(idx, idx)]

# 3.2 Candidate weight combos: WP / WS / WQ only + WQ-anchored pairs only -----------------------

W_dict = {'WP': WP, 'WS': WS, 'WQ': WQ}

def make_tasks_add_single_pair(W_dict_local):
    """
    Single-channel: WP, WS, WQ
    Pairs: keep only WQ+WP and WQ+WS (exclude WP+WS)
    """
    keys = list(W_dict_local.keys())
    tasks = []
    # Single-channel
    for k in keys:
        tasks.append((f'SEM_add_1_{k}', [W_dict_local[k]], 'add'))

    # Two-channel: allow only WQ+WP / WQ+WS
    if ('WP' in keys) and ('WQ' in keys):
        tasks.append(('SEM_add_2_WP_WQ',
                      [W_dict_local['WP'], W_dict_local['WQ']],
                      'add'))
    if ('WS' in keys) and ('WQ' in keys):
        tasks.append(('SEM_add_2_WS_WQ',
                      [W_dict_local['WS'], W_dict_local['WQ']],
                      'add'))
    return tasks

TASKS_MAIN_ADD = make_tasks_add_single_pair(W_dict)

# 3.3 Stage-1: US 2019 in-sample ranking --------------------------------------

base_rows = []

# OLS baseline (for readability; also compute Moran's I)
ols = fit_ols(y, X)
I0, p0 = moran_I_val_from_array(ols['resid'], WQ, ids=IDs)
base_rows.append(['OLS', '-', 'OLS',
                  ols['logL'], ols['AICc'], I0, p0])

# All single-/two-channel SEMs
for name, Wset, combo in TASKS_MAIN_ADD:
    res = fit_sem_multi(y, X, Wset, combo=combo)
    I, pI = moran_I_val_from_array(res['resid'], WQ, ids=IDs)
    wmix = ';'.join(name.split('_')[3:])
    base_rows.append([name, wmix, combo,
                      res['logL'], res['AICc'], I, pI])

base_df = pd.DataFrame(
    base_rows,
    columns=['Model', 'W_mix', 'Combo', 'logL', 'AICc', 'MoranI', 'p_norm']
)
base_df = base_df.sort_values(['Combo', 'AICc'])

print(f"\n=== Stage-1: In-sample ranking for {outcome_tag} 2019 (US) ===")
print(base_df.head(10))

# 3.4 Stage-2: LOSO (by STATE, on shortlist models)----------------------------

stage1_sem = base_df[(base_df['Model'] != 'OLS') &
                     (base_df['Combo'] == 'add')].copy()

single_pool = stage1_sem[stage1_sem['Model'].str.match(r'^SEM_add_1_')]
best_single_name = single_pool.nsmallest(1, 'AICc')['Model'].iloc[0]

TOP_K_PAIR = 5  # Not many pairs here; 5 is sufficient
pair_pool = stage1_sem[stage1_sem['Model'].str.match(r'^SEM_add_2_')]
pair_shortlist = pair_pool.nsmallest(TOP_K_PAIR, 'AICc')['Model'].tolist()

stage2_models = [best_single_name] + pair_shortlist

print(f"\n[Stage-1] Shortlist for LOSO (best single + TOP_{TOP_K_PAIR} pairs):")
print(stage2_models)


def _loso_state_worker(st, IDs_all, df_all, y_all, X_all, tasks,
                       WQ_array, XY_all_, K_):
    """
    One fold (one STATE) leave-one-state-out:
    - fit on the training set;
    - evaluate logL, AICc, MoranI on the holdout state.
    """
    hold_ids = df_all.loc[df_all['STATE'] == st, 'FIPS'].tolist()
    hold_set = set(hold_ids)

    hold_idx = np.array(
        [i for i, idd in enumerate(IDs_all) if idd in hold_set],
        dtype=int
    )
    train_idx = np.array(
        [i for i in range(len(IDs_all)) if i not in hold_set],
        dtype=int
    )

    out_rows = []

    # Skip tiny states
    if len(hold_idx) < 5 or len(train_idx) < (K_ + 1 + 2):
        return out_rows

    # OLS baseline (record logL only)
    ols_tr = fit_ols(y_all[train_idx], X_all[train_idx, :])
    out_rows.append([st, 'OLS', '-', 'OLS',
                     ols_tr['logL'], np.nan, np.nan, np.nan])

    for name, Wset, combo in tasks:
        # Training
        y_tr = y_all[train_idx]
        X_tr = X_all[train_idx, :]
        W_tr = [submatrix(Wk, train_idx) for Wk in Wset]
        res_tr = fit_sem_multi(y_tr, X_tr, W_tr, combo=combo)

        # holdout
        y_te = y_all[hold_idx]
        X_te = X_all[hold_idx, :]
        W_te = [submatrix(Wk, hold_idx) for Wk in Wset]
        N_te, K_te = X_te.shape
        m = len(W_te)

        A_te, logdetA_te = build_A_and_logdet(res_tr['lam'], W_te, combo=combo)
        if A_te is None:
            out_rows.append([
                st, name, ';'.join(name.split('_')[3:]), combo,
                -np.inf, np.inf, np.nan, np.nan
            ])
            continue

        e_te = y_te - X_te.dot(res_tr['beta'])
        v_te = A_te.dot(e_te)
        sse = float(v_te.T.dot(v_te))

        df_ = N_te - (K_te + m)
        sigma2 = sse/df_ if df_ > 0 else sse / max(N_te, 1)
        logL = (logdetA_te
                - (N_te/2)*np.log(2*np.pi*sigma2)
                - sse/(2*sigma2))
        AIC  = -2*logL + 2*(K_te + m)
        AICc = (AIC + (2*(K_te+m)*((K_te+m)+1)) /
                (N_te-(K_te+m)-1) if (N_te-(K_te+m)-1) > 0 else np.nan)

        # MoranI: fix connectivity on the holdout state's WQ before evaluation
        WQ_te = submatrix(WQ_array, hold_idx)
        XY_te = XY_all_[hold_idx, :]
        WQ_eval = ensure_connected_within_state(WQ_te, XY_te)
        mi = Moran(
            v_te,
            util.full2W(WQ_eval, ids=[IDs_all[i] for i in hold_idx]),
            permutations=0
        )

        out_rows.append([
            st, name, ';'.join(name.split('_')[3:]), combo,
            logL, AICc, mi.I, mi.p_norm
        ])

    return out_rows


states = sorted(df['STATE'].unique())
tasks_for_loso = [t for t in TASKS_MAIN_ADD if t[0] in stage2_models]

res_lists = Parallel(
    n_jobs=N_JOBS, backend='loky', verbose=0
)(
    delayed(_loso_state_worker)(
        st, IDs, df, y, X, tasks_for_loso, WQ, XY_all, K
    )
    for st in states
)

cv_loso = pd.DataFrame(
    [row for rows in res_lists for row in rows],
    columns=['STATE','Model','W_mix','Combo',
             'pred_logL','pred_AICc','pred_MoranI','pred_p']
)

print(f"\n=== Stage-2: LOSO (shortlist, US, outcome={outcome_tag}) — preview ===")
print(cv_loso.head(10))

# 3.5 Forward gate：strict / lenient / super_lenient -------------------------

def summarize_models(cv_df, prefix='SEM_add_'):
    sub = cv_df[cv_df['Model'].str.startswith(prefix)].copy()
    g = (sub.groupby('Model', as_index=False)
           .agg(pred_AICc_mean=('pred_AICc','mean'),
                pred_AICc_sd  =('pred_AICc','std'),
                absMI_mean    =('pred_MoranI',
                                 lambda s: s.abs().mean()),
                absMI_sd      =('pred_MoranI',
                                 lambda s: s.abs().std())))
    return g.dropna(subset=['pred_AICc_mean','absMI_mean'])


sum_tbl = summarize_models(cv_loso, prefix='SEM_add_')
sum_tbl['dAICc_vs_best'] = (
    sum_tbl['pred_AICc_mean'] - sum_tbl['pred_AICc_mean'].min()
)

best_by_AICc_row  = sum_tbl.loc[sum_tbl['pred_AICc_mean'].idxmin()]
best_by_AICc_name = best_by_AICc_row['Model']

# Gate profiles
GATE_PROFILE        = 'super_lenient'   # ★ Recommended: super_lenient
PREFER_PAIR_IF_TIED = True
TIE_DELTA_REL       = 0.01             # Treat relative AICc difference ≤1% as a tie

if GATE_PROFILE == 'strict':
    GATE_DAICC_REL_MAX       = -0.01
    GATE_MI_REL_IMPROVE_MIN  = 0.00
    GATE_WINRATE_MIN         = 0.70
elif GATE_PROFILE == 'lenient':
    GATE_DAICC_REL_MAX       = 0.00
    GATE_MI_REL_IMPROVE_MIN  = 0.05
    GATE_WINRATE_MIN         = 0.50
elif GATE_PROFILE == 'super_lenient':
    GATE_DAICC_REL_MAX       = 0.02
    GATE_MI_REL_IMPROVE_MIN  = -0.10
    GATE_WINRATE_MIN         = 0.10
else:
    raise ValueError("Unknown GATE_PROFILE")

best_single_row = sum_tbl[sum_tbl['Model'] == best_single_name].iloc[0]
pair_tbl = sum_tbl[sum_tbl['Model'].str.match(r'^SEM_add_2_')]

if len(pair_tbl) > 0:
    best_pair_row = pair_tbl.nsmallest(1, 'pred_AICc_mean').iloc[0]

    # ΔAICc (absolute & relative)
    dAICc = (best_pair_row['pred_AICc_mean'] -
             best_single_row['pred_AICc_mean'])
    AICc_single = best_single_row['pred_AICc_mean']
    dAICc_rel = dAICc / max(abs(AICc_single), 1e-9)

    # Relative improvement in Moran's I
    absMI_single = best_single_row['absMI_mean']
    absMI_pair   = best_pair_row['absMI_mean']
    rel_mi_gain  = (
        (absMI_single - absMI_pair) /
        max(absMI_single, 1e-9)
    )

    # State-level win_rate: proportion of states where the pair has smaller AICc
    A_ = cv_loso[cv_loso['Model'] == best_pair_row['Model']][
        ['STATE','pred_AICc']
    ].rename(columns={'pred_AICc':'pair'})
    B_ = cv_loso[cv_loso['Model'] == best_single_name][
        ['STATE','pred_AICc']
    ].rename(columns={'pred_AICc':'single'})
    J = A_.merge(B_, on='STATE', how='inner')
    win_rate = float((J['pair'] < J['single']).mean()) if len(J) > 0 else 0.0

    # Gate decision
    pass_gate = (
        (dAICc_rel <= GATE_DAICC_REL_MAX) and
        (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN) and
        (win_rate >= GATE_WINRATE_MIN)
    )

    if pass_gate:
        final_model_main = best_pair_row['Model']
        gate_note = (
            f"[{GATE_PROFILE}] Accepted 2-channel by gates: "
            f"ΔAICc={dAICc:.2f}, ΔAICc_rel={dAICc_rel:.3f}<= {GATE_DAICC_REL_MAX}, "
            f"rel|I|↓={rel_mi_gain:.3f}>= {GATE_MI_REL_IMPROVE_MIN}, "
            f"win_rate={win_rate:.2f}>= {GATE_WINRATE_MIN}."
        )
    else:
        # If the pair fails the gate but ΔAICc_rel is very small, trigger tie-preference
        if (GATE_PROFILE == 'super_lenient' and
            PREFER_PAIR_IF_TIED and
            (dAICc_rel <= TIE_DELTA_REL) and
            (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN) and
            (win_rate >= GATE_WINRATE_MIN/2)):
            final_model_main = best_pair_row['Model']
            gate_note = (
                f"[{GATE_PROFILE}] Tie-preference to 2-channel "
                f"(ΔAICc_rel={dAICc_rel:.3f}<= {TIE_DELTA_REL}); "
                f"rel|I|↓={rel_mi_gain:.3f}, win_rate={win_rate:.2f}."
            )
        else:
            final_model_main = best_single_name
            gate_note = (
                f"[{GATE_PROFILE}] Kept best single: pair failed gates "
                f"and tie-preference not triggered. "
                f"ΔAICc={dAICc:.2f}, ΔAICc_rel={dAICc_rel:.3f}, "
                f"rel|I|↓={rel_mi_gain:.3f}, win_rate={win_rate:.2f}."
            )
else:
    final_model_main = best_single_name
    dAICc = np.nan
    dAICc_rel = np.nan
    rel_mi_gain = np.nan
    win_rate = np.nan
    gate_note = f"[{GATE_PROFILE}] No eligible two-channel; kept best single."

print(f"\n[Final model for {outcome_tag} 2019, US] {final_model_main}")
print("[Gate note] ", gate_note)
print("[Best-by-AICc on shortlist] ", best_by_AICc_name)

# ============================================
# 9. Add vs Prod sensitivity (US, current STD outcome)
#    — On the same W set as the final add model, compare additive vs multiplicative A(λ) structure
# ============================================

RUN_PROD_SENS = True  # Enable/disable add vs prod sensitivity analysis

if RUN_PROD_SENS:
    # 9.1 Parse channel keys from the model name (WP / WS / WQ ...)
    def parse_W_keys_from_model(model_name, W_dict_local):
        """
        For example:
          'SEM_add_1_WS'      -> ['WS']
          'SEM_add_2_WS_WQ'   -> ['WS','WQ']
        Keep only keys that exist in W_dict_local.
        """
        parts = model_name.split('_')
        chans = [p for p in parts[3:] if p in W_dict_local]
        if len(chans) == 0:
            raise ValueError(f"Cannot parse channels from model name: {model_name}")
        return chans

    W_keys_final = parse_W_keys_from_model(final_model_main, W_dict)
    Wset_final   = [W_dict[k] for k in W_keys_final]

    # 9.2 Fit add and prod structures using the same W set

    # add structure (main spec; refit full-sample for consistent output)
    res_add_full = fit_sem_multi(y, X, Wset_final, combo='add')
    I_add_full, p_add_full = moran_I_val_from_array(
        res_add_full['resid'], WQ, ids=IDs
    )

    # prod structure (sensitivity check)
    res_prod_full = fit_sem_multi(y, X, Wset_final, combo='prod')
    I_prod_full, p_prod_full = moran_I_val_from_array(
        res_prod_full['resid'], WQ, ids=IDs
    )

    # 9.3 Assemble results table & save
    sens_df = pd.DataFrame([
        {
            'Outcome_tag': outcome_tag,
            'Model_name': final_model_main,
            'Combo': 'add',
            'W_mix': ';'.join(W_keys_final),
            'logL': res_add_full['logL'],
            'AICc': res_add_full['AICc'],
            'MoranI': I_add_full,
            'absMoranI': abs(I_add_full),
            'p_norm': p_add_full
        },
        {
            'Outcome_tag': outcome_tag,
            'Model_name': final_model_main.replace('SEM_add', 'SEM_prod'),
            'Combo': 'prod',
            'W_mix': ';'.join(W_keys_final),
            'logL': res_prod_full['logL'],
            'AICc': res_prod_full['AICc'],
            'MoranI': I_prod_full,
            'absMoranI': abs(I_prod_full),
            'p_norm': p_prod_full
        }
    ])

    print("\n=== Add vs Prod (sensitivity on final W-combo, US) ===")
    print(sens_df)

    suffix_gate = f"_{GATE_PROFILE}" if 'GATE_PROFILE' in globals() else ""
    out_name = f"APPENDIX_add_vs_prod_sensitivity_US_{outcome_tag}{suffix_gate}.csv"
    sens_df.to_csv(out_name, index=False)

    print(f"\n[Add vs Prod sensitivity] Saved: {out_name}")

else:
    print("\n[Add vs Prod sensitivity] RUN_PROD_SENS = False, skipped.")


### Covariate sensitivity

In [ ]:
# ============================================================
#  SEM (multi-W, US) - Fast Covariate Robustness Check
#
#  Goal:
#    Covariate sensitivity (US-wide), mirroring the Deep South logic:
#      - 9 baseline covariates
#      - 4 "socio-sensitive" vars: Poverty, Noinsuranc, crime16to1, Dissimilar
#      - 11 covariate profiles:
#          * full (all 9)
#          * drop each of the 4 sensitive vars (x4)
#          * drop pairs of the 4 sensitive vars (C(4,2) = 6)
#
#  Data & W-matrices:
#    - US_HIV_Merged_total_final.shp (same as main US analysis)
#    - US_County_PCI_2019.csv      -> WP (Twitter PCI)
#    - processed_sci_summary.csv   -> WS (Facebook SCI)
#    - Queen contiguity            -> WQ (adjacency)
#
#  Channel search space (consistent with US main SEM analysis):
#    - Singles: WP, WS, WQ
#    - Pairs:   WP+WQ, WS+WQ    (NO WP+WS)
#
#  Output (CSV):
#    - COV_US_in_sample_add_<cov_profile>_<GATE_PROFILE>.csv
#    - COV_US_final_decision_allProfiles_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_model_frequency_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_Wmix_frequency_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_lambda_table_<outcome>_<GATE_PROFILE>.csv
#
#  Note:
#    - No LOSO / cross-validation here (too expensive for 11 profiles).
#    - Only full-sample fits per profile for speed.
# ============================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from itertools import combinations
from numpy.linalg import slogdet
from scipy.optimize import minimize
from esda.moran import Moran
from libpysal.weights import util, W, Queen

np.seterr(all="ignore")
RANDOM_SEED = 7
rng = np.random.default_rng(RANDOM_SEED)

# ----------------------------
# Mode / gate configs
# ----------------------------
RUN_PROD = False  # keep for compatibility; not really used here

# Gate profile for pair vs single (in-sample only)
GATE_PROFILE = "lenient"          # options: "strict" or "lenient"
PREFER_PAIR_IF_TIED = True
TIE_DELTA = 2.0                   # AICc "tie" width (pair - single <= TIE_DELTA)

if GATE_PROFILE == "strict":
    # Pair must clearly outperform single (in-sample)
    GATE_DAICC_THRESH = -4.0      # ΔAICc = pair - single (negative = better)
    GATE_MI_REL_IMPROVE_MIN = 0.00  # |I| at least not worse
else:
    # Lenient: allow small AICc disadvantage if Moran's I improves
    GATE_DAICC_THRESH = +1.0      # allow pair up to +1 AICc worse
    GATE_MI_REL_IMPROVE_MIN = 0.02  # ≥ 2% reduction in |I|


# ============================================================
# A. Read US shapefile & basic preprocessing
# ============================================================

shp_path = "US_HIV_Merged_total_final.shp"
gdf = gpd.read_file(shp_path)

if "FIPS" not in gdf.columns:
    raise KeyError("Shapefile needs a 'FIPS' field named 'FIPS'.")

# Clean FIPS: strip leading zeros, drop duplicates
gdf["FIPS"] = gdf["FIPS"].astype(str).str.lstrip("0")
gdf = gdf.drop_duplicates(subset=["FIPS"])

# Fix possibly truncated 2018 variable names (for consistency; not directly used here)
rename_map = {
    "RateChlamy": "RateChlamydia2018",
    "RateGonorr": "RateGonorrhea2018",
    "RateHIV201": "RateHIV2018"
}
for short, long_name in rename_map.items():
    if (short in gdf.columns) and (long_name not in gdf.columns):
        gdf = gdf.rename(columns={short: long_name})

# Baseline covariates (same 9 as in main US analysis)
X_cols = [
    "Population", "UrbanRural", "Female", "Old", "Black",
    "Noinsuranc", "Poverty", "crime16to1", "Dissimilar"
]

cols_keep = [
    "FIPS",
    "GonorrheaR", "ChlamydiaR", "PercenHIVp",
    *X_cols,
    "RateChlamydia2018", "RateGonorrhea2018", "RateHIV2018",
    "geometry"
]

df_geo = gdf[cols_keep].copy()
df_geo = gpd.GeoDataFrame(df_geo, geometry="geometry", crs=gdf.crs)

# STATE code from FIPS
def fips2state(x):
    return str(x).zfill(5)[:2]

df_geo["STATE"] = df_geo["FIPS"].apply(fips2state)


# ============================================================
# B. Build spatial weights (WP / WS / WQ) — US-wide
# ============================================================

def subset_W(w: W, keep_ids):
    """
    Subset a libpysal W to a given id set, preserving the given ID order.
    """
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))
    keep = set(keep_ids)
    new_neighbors, new_weights = {}, {}
    for i in w.id_order:
        if i not in keep:
            continue
        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])
        kn, kw = [], []
        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kn.append(nb)
                kw.append(wt)
        new_neighbors[i] = kn
        new_weights[i] = kw
    return W(new_neighbors, new_weights, ids=keep_ids)


# 1) WP: Twitter PCI mobility
pci = pd.read_csv("US_County_PCI_2019.csv")
pci["place_i"] = pci["place_i"].astype(str).str.lstrip("0")
pci["place_j"] = pci["place_j"].astype(str).str.lstrip("0")

pci_f = pci[
    pci["place_i"].isin(df_geo["FIPS"])
    & pci["place_j"].isin(df_geo["FIPS"])
].copy()

A_pci = pci_f.pivot_table(
    index="place_i", columns="place_j",
    values="pci", fill_value=0
)
np.fill_diagonal(A_pci.values, 0.0)
A = A_pci.values.astype(float)
A = A + A.T - np.diag(np.diag(A))  # symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP_full = A / row_sum
wP = util.full2W(WP_full, ids=list(A_pci.index))

# 2) WS: Facebook SCI ties
sci = pd.read_csv("processed_sci_summary.csv")
sci["user_loc"] = sci["user_loc"].astype(str).str.lstrip("0")
sci["tfr_loc"]  = sci["tfr_loc"].astype(str).str.lstrip("0")

sci_f = sci[
    sci["user_loc"].isin(df_geo["FIPS"])
    & sci["tfr_loc"].isin(df_geo["FIPS"])
].copy()

A_sci = sci_f.pivot_table(
    index="user_loc", columns="tfr_loc",
    values="tscaled_sci", fill_value=0
)
np.fill_diagonal(A_sci.values, 0.0)
B = A_sci.values.astype(float)
B = B + B.T - np.diag(np.diag(B))
row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WS_full = B / row_sum
wS = util.full2W(WS_full, ids=list(A_sci.index))

# 3) WQ: queen contiguity (row-standardized)
g3857 = df_geo.to_crs(epsg=3857)
wQ = Queen.from_dataframe(g3857, ids=list(df_geo["FIPS"]))
wQ.transform = "R"  # row-standardize

# Harmonize ID set & ordering across df_geo, WP, WS, WQ
common = (
    set(df_geo["FIPS"])
    & set(wP.id_order)
    & set(wS.id_order)
    & set(wQ.id_order)
)

df_geo = df_geo[df_geo["FIPS"].isin(common)].copy()
wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

df_geo = df_geo.set_index("FIPS").loc[wQ.id_order].reset_index()
IDs = list(wQ.id_order)
STATE = df_geo["STATE"].values

WP, _ = wP.full()
WS, _ = wS.full()
WQ, _ = wQ.full()


# ============================================================
# C. Outcome & covariate profiles (US-wide)
# ============================================================

# Outcome: HIV prevalence (%)
y_col = "PercenHIVp"   # can be switched to 'GonorrheaR' / 'ChlamydiaR' if needed
outcome_tag = {
    "PercenHIVp": "HIV",
    "GonorrheaR": "Gonorrhea",
    "ChlamydiaR": "Chlamydia"
}.get(y_col, y_col)

# Full 9 covariates (same as X_cols above)
X_cols_full = X_cols.copy()

# Socioeconomic vars that reviewers are most concerned about
SOCIO_SENSITIVE = ["Poverty", "Noinsuranc", "crime16to1", "Dissimilar"]


def build_profile(base_vars, drop_vars):
    drop_set = set(drop_vars)
    return [v for v in base_vars if v not in drop_set]


# Generate 11 covariate profiles: full + drop 1 + drop 2 (Deep South logic)
COV_PROFILES = {}

# 1) full model (9 covariates)
COV_PROFILES["full"] = X_cols_full

# 2) drop single sensitive variable
for v in SOCIO_SENSITIVE:
    name = f"drop_{v}"
    COV_PROFILES[name] = build_profile(X_cols_full, [v])

# 3) drop pairs of sensitive variables
for v1, v2 in combinations(SOCIO_SENSITIVE, 2):
    name = f"drop_{v1}_{v2}"
    COV_PROFILES[name] = build_profile(X_cols_full, [v1, v2])


# ============================================================
# D. OLS / SEM / Moran helpers
# ============================================================

def fit_ols(y, X):
    """
    Standard OLS fit.
    Returns beta, residuals, log-likelihood, and AICc.
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)
    sse = float(e.T.dot(e))
    N, K = X.shape
    df_ = N - K
    sigma2 = sse / df_
    logL = (
        -(N / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )
    AIC = -2 * logL + 2 * K
    AICc = (
        AIC + (2 * K * (K + 1)) / (N - K - 1)
        if (N - K - 1) > 0 else np.nan
    )
    return {
        "model": "OLS",
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": e
    }


def build_A_and_logdet(lam, W_list, combo="add"):
    """
    Build A and log|A| for SEM:
      y = Xβ + u, (I - Σ λ_k W_k) u = ε.
    """
    N = W_list[0].shape[0]
    I = np.eye(N)

    if combo == "add":
        A = I.copy()
        for i, Wk in enumerate(W_list):
            A -= lam[i] * Wk
        sign, logdetA = slogdet(A)
        if sign <= 0:
            return None, None
        return A, logdetA

    elif combo == "prod":
        # Not used here, kept for compatibility
        A = I.copy()
        logdet_sum = 0.0
        for i, Wk in enumerate(W_list):
            Mi = I - lam[i] * Wk
            sign_i, logdet_i = slogdet(Mi)
            if sign_i <= 0:
                return None, None
            logdet_sum += logdet_i
            A = Mi @ A
        return A, logdet_sum

    else:
        raise ValueError("combo must be 'add' or 'prod'")


def neglog_sem(params, y, X, W_list, combo="add"):
    """
    Negative log-likelihood of SEM for optimization.
    params = [λ_1,...,λ_m, β_0,...,β_{K-1}]
    """
    N, K = X.shape
    m = len(W_list)
    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list, combo=combo)
    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)
    sse = float(v.T.dot(v))

    df_ = N - (K + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = (
        logdetA
        - (N / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )
    return -logL


def fit_sem(y, X, W_list, method="L-BFGS-B", combo="add"):
    """
    Single-start ML SEM fit (sufficient for covariate robustness).
    """
    N, K = X.shape
    m = len(W_list)

    # Starting point: OLS beta, λ = 0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]

    bounds = [(-0.99, 0.99)] * m + [(-np.inf, np.inf)] * K

    res = minimize(
        neglog_sem, x0,
        args=(y, X, W_list, combo),
        method=method,
        bounds=bounds,
        options={"maxiter": 400, "ftol": 1e-5, "gtol": 1e-5},
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K

    AIC = -2 * logL + 2 * p
    AICc = (
        AIC + (2 * p * (p + 1)) / (N - p - 1)
        if (N - p - 1) > 0 else np.nan
    )

    A, _ = build_A_and_logdet(lam, W_list, combo=combo)
    v = A.dot(y - X.dot(beta))

    return {
        "model": f"SEM_{combo}_{m}W",
        "lam": lam,
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": v,
        "combo": combo,
        "success": res.success,
    }


def moran_I_val_from_array(resid, W_array, ids):
    """
    Compute Moran's I using a numpy W-array and ID order.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)
    return mi.I, mi.p_norm


# ============================================================
# E. Candidate SEM tasks (single + pair, add) — US channel set
# ============================================================

CHANNEL_KEYS = ["WP", "WS", "WQ"]


def make_tasks_add_single_pair_US(W_dict_local):
    """
    Candidate SEM models (consistent with main US analysis):
      - Singles: WP, WS, WQ
      - Pairs:   WP+WQ, WS+WQ
      (No WP+WS pair)
    """
    tasks = []
    # singles
    for k in ["WP", "WS", "WQ"]:
        if k in W_dict_local:
            tasks.append((f"SEM_add_1_{k}", [W_dict_local[k]], "add"))

    # pairs: WP+WQ / WS+WQ if both exist
    if "WP" in W_dict_local and "WQ" in W_dict_local:
        tasks.append(("SEM_add_2_WP_WQ",
                      [W_dict_local["WP"], W_dict_local["WQ"]], "add"))
    if "WS" in W_dict_local and "WQ" in W_dict_local:
        tasks.append(("SEM_add_2_WS_WQ",
                      [W_dict_local["WS"], W_dict_local["WQ"]], "add"))

    return tasks


def extract_W_mix_from_model(name: str) -> str:
    """
    Extract W-mix part from a model name, e.g.
      'SEM_add_2_WP_WQ' -> 'WP_WQ'
    """
    parts = name.split("_")
    if len(parts) >= 4:
        return "_".join(parts[3:])
    return ""


# ============================================================
# F. Main loop over covariate profiles (US-wide)
# ============================================================

all_decisions = []
lambda_rows = []

for cov_name, X_cols_profile in COV_PROFILES.items():
    print("\n" + "=" * 80)
    print(f"Running US covariate profile: {cov_name}  "
          f"(#{len(X_cols_profile)} covariates)")
    print("X_cols =", X_cols_profile)
    print("=" * 80)

    # 1) Subset data for this profile: require outcome + covariates not NA
    df_sub = df_geo[["FIPS", y_col] + X_cols_profile + ["STATE", "geometry"]].dropna().copy()
    fips_sub = df_sub["FIPS"].tolist()
    keep_set = set(fips_sub)

    # indices in original ID order
    idx_sub = [i for i, idd in enumerate(IDs) if idd in keep_set]
    if len(idx_sub) == 0:
        print(f"[WARN] cov_profile={cov_name} -> empty sample; skip.")
        continue

    IDs_sub = [IDs[i] for i in idx_sub]

    def submatrix(M, idx):
        return M[np.ix_(idx, idx)]

    WP_sub = submatrix(WP, idx_sub)
    WS_sub = submatrix(WS, idx_sub)
    WQ_sub = submatrix(WQ, idx_sub)

    # Reorder df_sub to match IDs_sub
    df_sub = df_sub.set_index("FIPS").loc[IDs_sub].reset_index()

    y = df_sub[y_col].values
    X_raw = df_sub[X_cols_profile].values
    N = X_raw.shape[0]
    X = np.hstack([np.ones((N, 1)), X_raw])
    K = X.shape[1]

    W_dict_local = {
        "WP": WP_sub,
        "WS": WS_sub,
        "WQ": WQ_sub,
    }
    TASKS_MAIN_ADD = make_tasks_add_single_pair_US(W_dict_local)

    # ----------------------------
    # Stage-1: in-sample OLS + SEM
    # ----------------------------
    base_rows = []

    # OLS baseline
    ols = fit_ols(y, X)
    I0, p0 = moran_I_val_from_array(ols["resid"], WQ_sub, ids=IDs_sub)
    base_rows.append(
        ["OLS", "-", "OLS", ols["logL"], ols["AICc"], I0, p0]
    )

    # SEM: all single + pair tasks
    for name, Wset, combo in TASKS_MAIN_ADD:
        res = fit_sem(y, X, Wset, combo=combo)
        I, pI = moran_I_val_from_array(res["resid"], WQ_sub, ids=IDs_sub)
        wmix = ";".join(name.split("_")[3:])
        base_rows.append(
            [name, wmix, combo, res["logL"], res["AICc"], I, pI]
        )

    base_df = pd.DataFrame(
        base_rows,
        columns=[
            "Model", "W_mix", "Combo",
            "logL", "AICc",
            "MoranI", "p_norm"
        ],
    )
    base_df = base_df.sort_values(["Combo", "AICc"])

    print(f"\n=== In-sample ranking (AICc) [US, cov={cov_name}, outcome={outcome_tag}] ===")
    print(base_df.head(10))

    # Save per-profile in-sample table
    base_path = f"COV_US_in_sample_add_{cov_name}_{outcome_tag}_{GATE_PROFILE}.csv"
    base_df.to_csv(base_path, index=False)
    print(f"Saved in-sample table for cov_profile={cov_name} -> {base_path}")

    # ----------------------------
    # Simple gate: best single vs best pair (in-sample)
    # ----------------------------
    stage1_sem = base_df[(base_df["Model"] != "OLS")].copy()
    single_pool = stage1_sem[stage1_sem["Model"].str.match(r"^SEM_add_1_")]
    pair_pool   = stage1_sem[stage1_sem["Model"].str.match(r"^SEM_add_2_")]

    if len(single_pool) == 0:
        print(f"[WARN] No single-channel SEM for cov_profile={cov_name}.")
        continue

    best_single_row = single_pool.nsmallest(1, "AICc").iloc[0]
    best_single_name = best_single_row["Model"]

    if len(pair_pool) > 0:
        best_pair_row = pair_pool.nsmallest(1, "AICc").iloc[0]
        best_pair_name = best_pair_row["Model"]

        dAICc = best_pair_row["AICc"] - best_single_row["AICc"]  # pair - single

        absMI_single = abs(best_single_row["MoranI"])
        absMI_pair   = abs(best_pair_row["MoranI"])
        rel_mi_gain  = (
            (absMI_single - absMI_pair) / max(absMI_single, 1e-9)
        )

        # In cov-robustness, "win_rate" is degenerate: 1 if pair has smaller AICc, else 0.
        win_rate = 1.0 if best_pair_row["AICc"] < best_single_row["AICc"] else 0.0

        pass_gate = (
            (dAICc <= GATE_DAICC_THRESH)
            and (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN)
        )

        if pass_gate:
            final_model = best_pair_name
            gate_note = (
                f"[{GATE_PROFILE}] Accepted 2-channel by in-sample gate "
                f"(cov={cov_name}, outcome={outcome_tag}): "
                f"ΔAICc={dAICc:.2f} <= {GATE_DAICC_THRESH}, "
                f"rel|I|↓={rel_mi_gain:.3f} >= {GATE_MI_REL_IMPROVE_MIN}."
            )
        else:
            if PREFER_PAIR_IF_TIED and (dAICc <= TIE_DELTA):
                final_model = best_pair_name
                gate_note = (
                    f"[{GATE_PROFILE}] Tie-preference to 2-channel "
                    f"(cov={cov_name}, outcome={outcome_tag}; "
                    f"ΔAICc={dAICc:.2f} <= {TIE_DELTA}); "
                    f"rel|I|↓={rel_mi_gain:.3f}."
                )
            else:
                final_model = best_single_name
                gate_note = (
                    f"[{GATE_PROFILE}] Kept best single (cov={cov_name}, outcome={outcome_tag}): "
                    f"pair failed gate / tie condition. "
                    f"ΔAICc={dAICc:.2f}, rel|I|↓={rel_mi_gain:.3f}."
                )
    else:
        final_model = best_single_name
        dAICc = np.nan
        rel_mi_gain = np.nan
        win_rate = np.nan
        gate_note = (
            f"[{GATE_PROFILE}] No 2-channel candidate (cov={cov_name}, outcome={outcome_tag}); "
            f"kept best single."
        )

    print(f"\n[Final (US)] cov_profile={cov_name}, outcome={outcome_tag} -> {final_model}")
    print("[Gate note] ", gate_note)

    # Record decision line for this profile
    decision_df = pd.DataFrame({
        "Outcome_tag": [outcome_tag],
        "Outcome_col": [y_col],
        "Cov_profile": [cov_name],
        "n_covariates": [len(X_cols_profile)],
        "Profile": [GATE_PROFILE],
        "Final_model": [final_model],
        "Final_W_mix": [extract_W_mix_from_model(final_model)],
        "Best_single": [best_single_name],
        "Decision": [gate_note],
        "AICc_final": [
            float(
                base_df.loc[base_df["Model"] == final_model, "AICc"].iloc[0]
            )
        ],
        "MoranI_final": [
            float(
                base_df.loc[base_df["Model"] == final_model, "MoranI"].iloc[0]
            )
        ],
        "AICc_best_single": [float(best_single_row["AICc"])],
        "MoranI_best_single": [float(best_single_row["MoranI"])],
        "dAICc_pair_vs_single": [dAICc],
        "Rel_MI_gain(pair_vs_single)": [rel_mi_gain],
        "Win_rate_pair_vs_single": [win_rate],
        "GATE_DAICC_THRESH": [GATE_DAICC_THRESH],
        "GATE_MI_REL_IMPROVE_MIN": [GATE_MI_REL_IMPROVE_MIN],
        "PREFER_PAIR_IF_TIED": [PREFER_PAIR_IF_TIED],
        "TIE_DELTA": [TIE_DELTA],
    })
    all_decisions.append(decision_df)

    # Record lambda for the final model (for robustness narrative)
    if final_model.startswith("SEM_add_1_"):
        _, Wkey = final_model.split("SEM_add_1_")
        Wset_final = [W_dict_local[Wkey]]
    elif final_model.startswith("SEM_add_2_"):
        parts = final_model.split("_")
        Wkey1, Wkey2 = parts[3], parts[4]
        Wset_final = [W_dict_local[Wkey1], W_dict_local[Wkey2]]
    else:
        Wset_final = []

    if len(Wset_final) > 0:
        res_final = fit_sem(y, X, Wset_final, combo="add")
        lam = res_final["lam"]
        lam_row = {
            "Outcome_tag": outcome_tag,
            "Outcome_col": y_col,
            "Cov_profile": cov_name,
            "Final_model": final_model,
            "Final_W_mix": extract_W_mix_from_model(final_model),
        }
        for i, val in enumerate(lam):
            lam_row[f"lambda_{i+1}"] = float(val)
        lambda_rows.append(lam_row)


# ============================================================
# G. Cross-profile summaries (US)
# ============================================================

if all_decisions:
    meta_df = pd.concat(all_decisions, ignore_index=True)
    meta_path = (
        f"COV_US_final_decision_allProfiles_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    meta_df.to_csv(meta_path, index=False)

    # Frequency of each specific final model
    model_freq = (
        meta_df.groupby("Final_model", as_index=False)
        .agg(
            n_profiles=("Cov_profile", "nunique"),
            mean_n_covariates=("n_covariates", "mean"),
            mean_AICc_final=("AICc_final", "mean"),
        )
        .sort_values("n_profiles", ascending=False)
    )
    model_freq_path = (
        f"COV_US_model_frequency_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    model_freq.to_csv(model_freq_path, index=False)

    # Frequency of each W-mix (aggregated across profiles)
    mix_freq = (
        meta_df.groupby("Final_W_mix", as_index=False)
        .agg(n_profiles=("Cov_profile", "nunique"))
        .sort_values("n_profiles", ascending=False)
    )
    mix_freq_path = (
        f"COV_US_Wmix_frequency_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    mix_freq.to_csv(mix_freq_path, index=False)

    # Lambda table (for robustness narrative)
    if lambda_rows:
        lam_df = pd.DataFrame(lambda_rows)
        lam_path = f"COV_US_lambda_table_{outcome_tag}_{GATE_PROFILE}.csv"
        lam_df.to_csv(lam_path, index=False)
    else:
        lam_path = "(none)"

    print("\n[Summary across covariate profiles, US]")
    print(f"  - Decisions per profile: {meta_path}")
    print(f"  - Model frequency:       {model_freq_path}")
    print(f"  - W-mix frequency:       {mix_freq_path}")
    print(f"  - Lambda table:          {lam_path}")
else:
    print("\n[INFO] No valid covariate profiles were estimated (US).")

###SLM vs SEM hero

In [ ]:
# ============================================
# OLS vs hero SEM vs hero-structure SLM (US)
# ============================================
import numpy as np
import pandas as pd
import geopandas as gpd

from numpy.linalg import slogdet
from scipy.optimize import minimize
from esda.moran import Moran
from libpysal.weights import util, W, Queen

np.seterr(all='ignore')

# --------------------------------------------
# 0. Global config
# --------------------------------------------

# Outcome column: HIV / Gonorrhea / Chlamydia
#   HIV:         y_col = 'PercenHIVp'
#   Gonorrhea:   y_col = 'GonorrheaR'
#   Chlamydia:   y_col = 'ChlamydiaR'
y_col = 'PercenHIVp'

# Corresponding hero SEM structure (use your finalized choice)
# Example:
#   HIV        : 'SEM_add_2_WP_WQ'
#   Gonorrhea  : 'SEM_add_1_WQ'
#   Chlamydia  : 'SEM_add_2_WS_WQ'
hero_model = 'SEM_add_2_WP_WQ'

OUTCOME_TAG = {
    'PercenHIVp': 'HIV',
    'GonorrheaR': 'Gonorrhea',
    'ChlamydiaR': 'Chlamydia'
}
outcome_tag = OUTCOME_TAG.get(y_col, y_col)

# File paths (change to match your project paths)
shp_path = "US_HIV_Merged_total_final.shp"
pci_path = "US_County_PCI_2019.csv"
sci_path = "processed_sci_summary.csv"

RANDOM_SEED = 7

# Covariates (keep consistent with your main analysis)
X_cols = [
    'Population', 'UrbanRural', 'Female', 'Old', 'Black',
    'Noinsuranc', 'Poverty', 'crime16to1', 'Dissimilar'
]

# --------------------------------------------
# 1. Read shapefile & basic cleaning
# --------------------------------------------
gdf = gpd.read_file(shp_path)

if 'FIPS' not in gdf.columns:
    raise KeyError("Shapefile must contain a 'FIPS' field.")

# Strip leading zeros from FIPS and drop duplicates
gdf['FIPS'] = gdf['FIPS'].astype(str).str.lstrip('0')
gdf = gdf.drop_duplicates(subset=['FIPS'])

# Check that outcome and covariate columns exist
cols_need = ['FIPS', y_col] + X_cols + ['geometry']
missing_cols = set(cols_need) - set(gdf.columns)
if missing_cols:
    raise KeyError(f"Shapefile is missing required fields: {missing_cols}")

df_geo = gdf[cols_need].copy()

# Drop missing values for the current outcome + covariates
df_geo = df_geo.dropna(subset=[y_col] + X_cols)

# State code (first two digits of FIPS)
def fips2state(x):
    return str(x).zfill(5)[:2]

df_geo['STATE'] = df_geo['FIPS'].apply(fips2state)
df_geo = gpd.GeoDataFrame(df_geo, geometry='geometry', crs=gdf.crs)

# --------------------------------------------
# 2. Construct three spatial weight matrices: WP / WS / WQ (US)
# --------------------------------------------

# 2.1 WP: Twitter PCI mobility
pci = pd.read_csv(pci_path)
pci['place_i'] = pci['place_i'].astype(str).str.lstrip('0')
pci['place_j'] = pci['place_j'].astype(str).str.lstrip('0')

pci_f = pci[
    pci['place_i'].isin(df_geo['FIPS']) &
    pci['place_j'].isin(df_geo['FIPS'])
].copy()

A_pci = pci_f.pivot_table(
    index='place_i', columns='place_j',
    values='pci', fill_value=0.0
)

# Zero diagonal, row-standardize
np.fill_diagonal(A_pci.values, 0.0)
A = A_pci.values.astype(float)
A = A + A.T - np.diag(np.diag(A))     # Symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP_full = A / row_sum                 # Row-standardize
wP = util.full2W(WP_full, ids=list(A_pci.index))

# 2.2 WS: Facebook SCI social ties
sci = pd.read_csv(sci_path)
sci['user_loc'] = sci['user_loc'].astype(str).str.lstrip('0')
sci['tfr_loc']  = sci['tfr_loc'].astype(str).str.lstrip('0')

sci_f = sci[
    sci['user_loc'].isin(df_geo['FIPS']) &
    sci['tfr_loc'].isin(df_geo['FIPS'])
].copy()

A_sci = sci_f.pivot_table(
    index='user_loc', columns='tfr_loc',
    values='tscaled_sci', fill_value=0.0
)

np.fill_diagonal(A_sci.values, 0.0)
B = A_sci.values.astype(float)
B = B + B.T - np.diag(np.diag(B))
row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WS_full = B / row_sum
wS = util.full2W(WS_full, ids=list(A_sci.index))

# 2.3 WQ: queen contiguity (US counties with data)
g3857 = df_geo.to_crs(epsg=3857)
wQ = Queen.from_dataframe(g3857, ids=list(df_geo['FIPS']))
wQ.transform = 'R'   # Row-standardized
WQ_full, _ = wQ.full()

# --------------------------------------------
# 3. Align ID set & ordering
#    —— ensure df_geo / WP / WS / WQ use the same FIPS ordering
# --------------------------------------------

common = (
    set(df_geo['FIPS']) &
    set(wP.id_order) &
    set(wS.id_order) &
    set(wQ.id_order)
)

def subset_W(w: W, keep_ids):
    """
    Trim W to contain only keep_ids, and preserve row-standardization.
    All W matrices share the same keep_ids order to guarantee consistency.
    """
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))  # fixed order
    keep = set(keep_ids)
    new_neighbors, new_weights = {}, {}

    for i in w.id_order:
        if i not in keep:
            continue
        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])
        kn, kw = [], []
        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kn.append(nb)
                kw.append(wt)
        new_neighbors[i] = kn
        new_weights[i] = kw

    return W(new_neighbors, new_weights, ids=keep_ids)

# Trim to common IDs
df_geo = df_geo[df_geo['FIPS'].isin(common)].copy()
wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

# Align df_geo ordering to wQ.id_order
df_geo = df_geo.set_index('FIPS').loc[wQ.id_order].reset_index()
IDs = list(wQ.id_order)

# Convert back to numpy matrices
WP, _       = wP.full()
WS, _       = wS.full()
WQ_array, _ = wQ.full()

# --------------------------------------------
# 4. Build y & X
# --------------------------------------------
y = df_geo[y_col].values
X_raw = df_geo[X_cols].values
N = X_raw.shape[0]

# Add intercept
X = np.hstack([np.ones((N, 1)), X_raw])
K = X.shape[1]

# --------------------------------------------
# 5. Core helper functions: OLS / SEM / SLM
# --------------------------------------------

def moran_I_val_from_array(resid, W_array, ids):
    """
    Given residuals and full W (numpy array), compute Moran's I.
    Use WQ as the unified evaluation weight matrix here.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)
    return mi.I, mi.p_norm


def fit_ols(y, X, W_eval, ids):
    """
    Standard OLS fit + residual Moran's I
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)
    sse = float(e.T.dot(e))
    N_, K_ = X.shape
    df_ = N_ - K_
    sigma2 = sse / df_
    logL = -(N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    AIC = -2*logL + 2*K_
    AICc = (AIC + (2*K_*(K_+1))/(N_-K_-1)
            if (N_-K_-1) > 0 else np.nan)

    I, pI = moran_I_val_from_array(e, W_eval, ids)
    return {
        'model_type': 'OLS',
        'W_mix': '-',
        'logL': logL,
        'AICc': AICc,
        'MoranI': I,
        'p_norm': pI
    }


def build_A_and_logdet(lam, W_list):
    """
    A = I - Σ λ_k W_k, return A and log|A|
    """
    N_ = W_list[0].shape[0]
    I = np.eye(N_)

    A = I.copy()
    for i, Wk in enumerate(W_list):
        A -= lam[i] * Wk

    sign, logdetA = slogdet(A)
    if sign <= 0:
        return None, None
    return A, logdetA


def neglog_sem(params, y, X, W_list):
    """
    Negative log-likelihood for multi-channel SEM (error model):
      y = Xβ + μ,  μ = (I - Σ λ_k W_k)^(-1) ε
      ⇒ ε = A (y - Xβ), logL includes log|A|
    params = [λ_1,...,λ_m, β_0,...,β_{K-1}]
    """
    N_, K_ = X.shape
    m = len(W_list)
    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list)
    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)
    sse = float(v.T.dot(v))

    df_ = N_ - (K_ + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = logdetA - (N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    return -logL


def fit_sem_multi(y, X, W_list):
    """
    Multi-network SEM fit (for the hero model).
    """
    N_, K_ = X.shape
    m = len(W_list)

    # Start: OLS beta + lambda=0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)]*m + [(-np.inf, np.inf)]*K_

    res = minimize(
        neglog_sem, x0,
        args=(y, X, W_list),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 200, 'ftol': 1e-4, 'gtol': 1e-4}
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K_
    AIC = -2*logL + 2*p
    AICc = AIC + (2*p*(p+1))/(N_-p-1) if (N_-p-1) > 0 else np.nan

    A, _ = build_A_and_logdet(lam, W_list)
    v = A.dot(y - X.dot(beta))  # whitened residual

    return lam, beta, logL, AICc, v


def neglog_slm(params, y, X, W_list):
    """
    Negative log-likelihood for multi-channel SLM (lag model):
      y = Σ ρ_k W_k y + Xβ + u, u ~ N(0, σ^2 I)
      Let A = I - Σ ρ_k W_k, then u = A y - Xβ
      logL = log|A| - (n/2)log(2πσ^2) - (u'u)/(2σ^2)
    params = [ρ_1,...,ρ_m, β_0,...,β_{K-1}]
    """
    N_, K_ = X.shape
    m = len(W_list)
    rho = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(rho, W_list)
    if A is None:
        return 1e12

    u = A.dot(y) - X.dot(beta)
    sse = float(u.T.dot(u))

    df_ = N_ - (K_ + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = logdetA - (N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    return -logL


def fit_slm_multi(y, X, W_list):
    """
    Multi-network SLM fit (hero lag benchmark).
    """
    N_, K_ = X.shape
    m = len(W_list)

    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)]*m + [(-np.inf, np.inf)]*K_

    res = minimize(
        neglog_slm, x0,
        args=(y, X, W_list),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 200, 'ftol': 1e-4, 'gtol': 1e-4}
    )

    params = res.x
    rho = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K_
    AIC = -2*logL + 2*p
    AICc = AIC + (2*p*(p+1))/(N_-p-1) if (N_-p-1) > 0 else np.nan

    A, _ = build_A_and_logdet(rho, W_list)
    u = A.dot(y) - X.dot(beta)  # model residual u

    return rho, beta, logL, AICc, u


# --------------------------------------------
# 6. Parse hero model name and extract W channels for SEM/SLM
# --------------------------------------------

W_dict = {
    'WP': WP,
    'WS': WS,
    'WQ': WQ_array
}

def get_W_list_from_model_name(model_name, W_dict_local):
    """
    Examples:
      'SEM_add_1_WS'    -> channels = ['WS']
      'SEM_add_2_WP_WQ' -> channels = ['WP','WQ']
    Extract 'WP'/'WS'/'WQ' in the order they appear in the name.
    """
    parts = model_name.split('_')
    chans = [p for p in parts if p in W_dict_local]
    if len(chans) == 0:
        raise ValueError(f"Cannot parse channels from model name: {model_name}")
    W_list = [W_dict_local[ch] for ch in chans]
    return W_list, chans

W_list_hero, hero_channels = get_W_list_from_model_name(hero_model, W_dict)
W_mix_str = '+'.join(hero_channels)

print(f"[Config] Outcome = {outcome_tag}, y_col = {y_col}")
print(f"[Config] Hero SEM structure = {hero_model} (channels: {W_mix_str})")
print(f"[Sample] N = {N} counties")

# --------------------------------------------
# 7. Fit three models: OLS / hero SEM / hero-structure SLM
# --------------------------------------------

# 7.1 OLS
ols_res = fit_ols(y, X, W_eval=WQ_array, ids=IDs)

# 7.2 hero SEM (error)
lam_sem, beta_sem, logL_sem, AICc_sem, resid_sem = fit_sem_multi(
    y, X, W_list_hero
)
I_sem, p_sem = moran_I_val_from_array(resid_sem, WQ_array, IDs)
sem_res = {
    'model_type': 'SEM_error',
    'W_mix': W_mix_str,
    'logL': logL_sem,
    'AICc': AICc_sem,
    'MoranI': I_sem,
    'p_norm': p_sem
}

# 7.3 hero-structure SLM (lag)
rho_slm, beta_slm, logL_slm, AICc_slm, resid_slm = fit_slm_multi(
    y, X, W_list_hero
)
I_slm, p_slm = moran_I_val_from_array(resid_slm, WQ_array, IDs)
slm_res = {
    'model_type': 'SLM_lag',
    'W_mix': W_mix_str,
    'logL': logL_slm,
    'AICc': AICc_slm,
    'MoranI': I_slm,
    'p_norm': p_slm
}

# --------------------------------------------
# 8. Assemble appendix table & save
# --------------------------------------------

res_table = pd.DataFrame([ols_res, sem_res, slm_res])

# Sort: AICc first, then |MoranI|
res_table = res_table.sort_values(
    by=['AICc', res_table['MoranI'].abs().name],
    ascending=[True, True]
).reset_index(drop=True)

print(f"\n=== OLS vs hero SEM vs hero-structure SLM (US, {outcome_tag}, 2019) ===")
print(res_table)

out_csv = f"APPENDIX_SEM_vs_SLM_US_{outcome_tag}.csv"
res_table.to_csv(out_csv, index=False)
print(f"\n[Saved] {out_csv}")